# B3 · Attention et Transformers : un mini-GPT from scratch — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b3_transformers/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`. Tout tourne sur CPU : la partie B entraîne ton mini-GPT en 2 à 3 min en mode rapide (🚀 environ 35 min en mode complet, plutôt sur un GPU de Colab), puis le sauvegarde hors du dépôt pour la suite du chapitre.

> Travaille dans **ta copie** (`mon_travail/b3_transformers/03_notebook.ipynb`, créée par `python tools/start_chapter.py B3`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B3.Q1 à B3.Q12, B3.R1 à B3.R3, B3.1 à B3.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B3.12 | causal_mask, padding_mask et scaled_dot_product_attention | 🔨 | ★★ | 30 |
|  | B3.13 | Prédire la carte d'attention quand une clé copie la requête, puis quand on retire la division par √d_k | 🔮 | ★ | 10 |
|  | B3.14 | split_heads, merge_heads et multi_head_attention | 🔨 | ★★★ | 45 |
|  | B3.15 | positional_encoding et apply_rope | 🔨 | ★★ | 30 |
|  | B3.16 | transformer_block en NumPy, validé contre nn.TransformerEncoderLayer | 🔨 | ★★★★ | 100 |
|  | B3.17 | Trois masques qui mentent : convention inversée, masque transposé, padding compté dans la loss | 🐛 | ★★ | 25 |
| B | B3.18 | De NumPy à PyTorch : un module CausalSelfAttention avec F.scaled_dot_product_attention | 📦 | ★★ | 30 |
|  | B3.19 | Mini-GPT caractère par caractère : embeddings de tokens et de positions, blocs pré-LN, tête liée | 📦 | ★★★ | 60 |
|  | B3.20 | Entraîner le mini-GPT sur Holmes : AdamW, warmup + cosinus, clipping, perplexité, face au LSTM du ch. 22 🚀 | 📦 | ★★★ | 60 |
| B | B3.21 à B3.30 | la suite du mini-GPT : courbes, Holmes amorcé en français, KV cache, cartes d'attention, ablations, BPE contre caractères, contexte long, ViT, test de causalité, défi (*prochaine étape de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B3")  # your own library

## Objectifs et rappel express

- Calculer à la main une attention par produit scalaire, avec et sans masque causal, et justifier la division par $\sqrt{d_k}$.
- Implémenter en NumPy les masques, l'attention multi-têtes, les encodages de position (sinusoïdal, RoPE) et un bloc Transformer pré-LN, et les valider contre PyTorch.
- Écrire et entraîner en PyTorch un mini-GPT sur Holmes, le faire générer avec un KV cache, et comparer sa perplexité à celle du LSTM du ch. 22.
- Estimer les paramètres, les FLOPs et la mémoire d'un Transformer, et mesurer le coût d'un long contexte.

**Rappel express.** $\operatorname{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \operatorname{softmax}\big(\mathbf{Q}\mathbf{K}^\top / \sqrt{d_k} + \mathbf{M}\big)\mathbf{V}$, la softmax sur les clés, $M_{ij} = -\infty$ là où le masque vaut `False`. Masque causal : la requête $i$ voit les clés $j \le i + S - L$. Multi-têtes : $(N, T, d) \to (N, H, T, d_h)$, $4d^2 + 4d$ paramètres. Encodage sinusoïdal : $PE_{p, 2i} = \sin(p\,\omega_i)$, $PE_{p, 2i+1} = \cos(p\,\omega_i)$, $\omega_i = 10000^{-2i/d}$. RoPE : $\langle R_{m\theta} \mathbf{q}, R_{n\theta} \mathbf{k} \rangle = \langle \mathbf{q}, R_{(n-m)\theta} \mathbf{k} \rangle$. Bloc pré-LN : $\mathbf{h} = \mathbf{x} + \operatorname{MHA}(\operatorname{LN}_1(\mathbf{x}))$, $\mathbf{y} = \mathbf{h} + \operatorname{MLP}(\operatorname{LN}_2(\mathbf{h}))$, environ $12d^2$ paramètres. Entraînement : environ $6ND$ FLOPs ; KV cache : $2 \times L \times T \times d \times$ octets par valeur.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B3.1 à B3.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une forme de tableau : entre guillemets (`"(2, 3)"`) ; un vecteur ou une ligne de matrice : une liste (`[0.5, -1, 3]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ B3.6 et B3.7, l'estimation 🧮 B3.8, la réflexion (🗣️ B3.9, ⚖️ B3.10, 📄 B3.11) et l'entretien se corrigent avec `05_solutions.md`.

**B3.Q1 — Le goulot de l'encodeur-décodeur récurrent : pourquoi un seul vecteur ne suffit pas**

In [ ]:
# B3.Q1: the values you found on paper
answer_B3_Q1a = ...  # a) the letter of your choice, e.g. "E"
answer_B3_Q1b = ...  # b) True or False
answer_B3_Q1c = ...  # c) a whole number
answer_B3_Q1d = ...  # d) the letter of your choice, e.g. "E"
answer_B3_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_Q1a, answer_B3_Q1b, answer_B3_Q1c, answer_B3_Q1d, answer_B3_Q1e]):
    wb.check(f"B3.Q1{letter}", answer)

**B3.Q2 — Requête, clé, valeur : un dictionnaire où l'on cherche « à peu près »**

In [ ]:
# B3.Q2: the values you found on paper
answer_B3_Q2a = ...  # a) the letter of your choice, e.g. "E"
answer_B3_Q2b = ...  # b) True or False
answer_B3_Q2c = ...  # c) a number, 2 decimals
answer_B3_Q2d = ...  # d) a number, 1 decimal
answer_B3_Q2e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B3_Q2a, answer_B3_Q2b, answer_B3_Q2c, answer_B3_Q2d, answer_B3_Q2e]):
    wb.check(f"B3.Q2{letter}", answer)

**B3.Q3 — Pourquoi diviser les scores par √d_k ?**

In [ ]:
# B3.Q3: the values you found on paper
answer_B3_Q3a = ...  # a) a whole number
answer_B3_Q3b = ...  # b) True or False
answer_B3_Q3c = ...  # c) the letter of your choice, e.g. "E"
answer_B3_Q3d = ...  # d) the letter of your choice, e.g. "E"
answer_B3_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_Q3a, answer_B3_Q3b, answer_B3_Q3c, answer_B3_Q3d, answer_B3_Q3e]):
    wb.check(f"B3.Q3{letter}", answer)

**B3.Q4 — Masque causal : qui a le droit de regarder qui ?**

In [ ]:
# B3.Q4: the values you found on paper
answer_B3_Q4a = ...  # a) a whole number
answer_B3_Q4b = ...  # b) the letter of your choice, e.g. "E"
answer_B3_Q4c = ...  # c) True or False
answer_B3_Q4d = ...  # d) a whole number
answer_B3_Q4e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B3_Q4a, answer_B3_Q4b, answer_B3_Q4c, answer_B3_Q4d, answer_B3_Q4e]):
    wb.check(f"B3.Q4{letter}", answer)

**B3.Q5 — Auto-attention ou attention croisée : d'où viennent Q, K et V ?**

In [ ]:
# B3.Q5: the values you found on paper
answer_B3_Q5a = ...  # a) the letter of your choice, e.g. "E"
answer_B3_Q5b = ...  # b) a shape as a string, e.g. "(2, 3)"
answer_B3_Q5c = ...  # c) True or False
answer_B3_Q5d = ...  # d) the letter of your choice, e.g. "E"
answer_B3_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_Q5a, answer_B3_Q5b, answer_B3_Q5c, answer_B3_Q5d, answer_B3_Q5e]):
    wb.check(f"B3.Q5{letter}", answer)

**B3.Q6 — Plusieurs têtes pour le prix d'une : que gagne-t-on ?**

In [ ]:
# B3.Q6: the values you found on paper
answer_B3_Q6a = ...  # a) a whole number
answer_B3_Q6b = ...  # b) True or False
answer_B3_Q6c = ...  # c) a whole number
answer_B3_Q6d = ...  # d) the letter of your choice, e.g. "E"
answer_B3_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_Q6a, answer_B3_Q6b, answer_B3_Q6c, answer_B3_Q6d, answer_B3_Q6e]):
    wb.check(f"B3.Q6{letter}", answer)

**B3.Q7 — Sans encodage de position, l'attention voit-elle l'ordre des mots ?**

In [ ]:
# B3.Q7: the values you found on paper
answer_B3_Q7a = ...  # a) True or False
answer_B3_Q7b = ...  # b) the letter of your choice, e.g. "E"
answer_B3_Q7c = ...  # c) the letter of your choice, e.g. "E"
answer_B3_Q7d = ...  # d) the letter of your choice, e.g. "E"
answer_B3_Q7e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B3_Q7a, answer_B3_Q7b, answer_B3_Q7c, answer_B3_Q7d, answer_B3_Q7e]):
    wb.check(f"B3.Q7{letter}", answer)

**B3.Q8 — Résiduel, pré-LN ou post-LN : ce qui rend un Transformer profond entraînable**

In [ ]:
# B3.Q8: the values you found on paper
answer_B3_Q8a = ...  # a) the letter of your choice, e.g. "E"
answer_B3_Q8b = ...  # b) True or False
answer_B3_Q8c = ...  # c) the letter of your choice, e.g. "E"
answer_B3_Q8d = ...  # d) a whole number
answer_B3_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_Q8a, answer_B3_Q8b, answer_B3_Q8c, answer_B3_Q8d, answer_B3_Q8e]):
    wb.check(f"B3.Q8{letter}", answer)

**B3.Q9 — BERT, GPT, T5 : quel masque, quel objectif, quel usage ?**

In [ ]:
# B3.Q9: the values you found on paper
answer_B3_Q9a = ...  # a) the letter of your choice, e.g. "E"
answer_B3_Q9b = ...  # b) a whole number
answer_B3_Q9c = ...  # c) a whole number
answer_B3_Q9d = ...  # d) True or False
answer_B3_Q9e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B3_Q9a, answer_B3_Q9b, answer_B3_Q9c, answer_B3_Q9d, answer_B3_Q9e]):
    wb.check(f"B3.Q9{letter}", answer)

**B3.Q10 — Décalage d'un token : ce que chaque position apprend à prédire**

In [ ]:
# B3.Q10: the values you found on paper
answer_B3_Q10a = ...  # a) the letter of your choice, e.g. "E"
answer_B3_Q10b = ...  # b) a whole number
answer_B3_Q10c = ...  # c) True or False
answer_B3_Q10d = ...  # d) the letter of your choice, e.g. "E"
answer_B3_Q10e = ...  # e) a number, 2 decimals

for letter, answer in zip("abcde", [answer_B3_Q10a, answer_B3_Q10b, answer_B3_Q10c, answer_B3_Q10d, answer_B3_Q10e]):
    wb.check(f"B3.Q10{letter}", answer)

**B3.Q11 — KV cache : ce que l'on garde et pourquoi la génération s'accélère**

In [ ]:
# B3.Q11: the values you found on paper
answer_B3_Q11a = ...  # a) the letter of your choice, e.g. "E"
answer_B3_Q11b = ...  # b) a whole number
answer_B3_Q11c = ...  # c) True or False
answer_B3_Q11d = ...  # d) the letter of your choice, e.g. "E"
answer_B3_Q11e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B3_Q11a, answer_B3_Q11b, answer_B3_Q11c, answer_B3_Q11d, answer_B3_Q11e]):
    wb.check(f"B3.Q11{letter}", answer)

**B3.Q12 — Coût quadratique du contexte et ce que change FlashAttention**

In [ ]:
# B3.Q12: the values you found on paper
answer_B3_Q12a = ...  # a) a whole number
answer_B3_Q12b = ...  # b) True or False
answer_B3_Q12c = ...  # c) the letter of your choice, e.g. "E"
answer_B3_Q12d = ...  # d) a whole number
answer_B3_Q12e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_Q12a, answer_B3_Q12b, answer_B3_Q12c, answer_B3_Q12d, answer_B3_Q12e]):
    wb.check(f"B3.Q12{letter}", answer)

**B3.R1 — De quoi dépend le nombre de tokens d'un texte ?**

In [ ]:
# B3.R1: the values you found on paper
answer_B3_R1a = ...  # a) True or False
answer_B3_R1b = ...  # b) a whole number
answer_B3_R1c = ...  # c) the letter of your choice, e.g. "E"
answer_B3_R1d = ...  # d) True or False
answer_B3_R1e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B3_R1a, answer_B3_R1b, answer_B3_R1c, answer_B3_R1d, answer_B3_R1e]):
    wb.check(f"B3.R1{letter}", answer)

**B3.R2 — Pourquoi le gradient s'évanouit dans un RNN déroulé**

In [ ]:
# B3.R2: the values you found on paper
answer_B3_R2a = ...  # a) a number, 4 decimals
answer_B3_R2b = ...  # b) True or False
answer_B3_R2c = ...  # c) the letter of your choice, e.g. "E"
answer_B3_R2d = ...  # d) the letter of your choice, e.g. "E"
answer_B3_R2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_R2a, answer_B3_R2b, answer_B3_R2c, answer_B3_R2d, answer_B3_R2e]):
    wb.check(f"B3.R2{letter}", answer)

**B3.R3 — Layer norm et connexion résiduelle, à quoi sert chacune ?**

In [ ]:
# B3.R3: the values you found on paper
answer_B3_R3a = ...  # a) a number, 3 decimals
answer_B3_R3b = ...  # b) True or False
answer_B3_R3c = ...  # c) the letter of your choice, e.g. "E"
answer_B3_R3d = ...  # d) True or False
answer_B3_R3e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B3_R3a, answer_B3_R3b, answer_B3_R3c, answer_B3_R3d, answer_B3_R3e]):
    wb.check(f"B3.R3{letter}", answer)

**Ex B3.1 — L'attention à la main : trois tokens, d_k = 2, scores, softmax et sortie**

In [ ]:
# Ex B3.1: the values you found on paper
answer_B3_1a = ...  # a) a list of numbers, e.g. [0.5, -1, 3]
answer_B3_1b = ...  # b) a number, 3 decimals
answer_B3_1c = ...  # c) a number, 3 decimals
answer_B3_1d = ...  # d) a list of numbers, 3 decimals each
answer_B3_1e = ...  # e) a number, 3 decimals
answer_B3_1f = ...  # f) True or False
answer_B3_1g = ...  # g) a number, 3 decimals

for letter, answer in zip("abcdefg", [answer_B3_1a, answer_B3_1b, answer_B3_1c, answer_B3_1d, answer_B3_1e, answer_B3_1f, answer_B3_1g]):
    wb.check(f"B3.1{letter}", answer)

**Ex B3.2 — Le même calcul avec un masque causal**

In [ ]:
# Ex B3.2: the values you found on paper
answer_B3_2a = ...  # a) a whole number
answer_B3_2b = ...  # b) a list of numbers, 3 decimals each
answer_B3_2c = ...  # c) a list of numbers, 3 decimals each
answer_B3_2d = ...  # d) a list of numbers, 3 decimals each
answer_B3_2e = ...  # e) True or False
answer_B3_2f = ...  # f) a whole number
answer_B3_2g = ...  # g) a whole number
answer_B3_2h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B3_2a, answer_B3_2b, answer_B3_2c, answer_B3_2d, answer_B3_2e, answer_B3_2f, answer_B3_2g, answer_B3_2h]):
    wb.check(f"B3.2{letter}", answer)

**Ex B3.3 — Suivre les formes dans l'attention multi-têtes et compter ses paramètres**

In [ ]:
# Ex B3.3: the values you found on paper
answer_B3_3a = ...  # a) a shape as a string, e.g. "(2, 3)"
answer_B3_3b = ...  # b) a shape as a string, e.g. "(2, 3)"
answer_B3_3c = ...  # c) a shape as a string, e.g. "(2, 3)"
answer_B3_3d = ...  # d) a whole number
answer_B3_3e = ...  # e) a whole number
answer_B3_3f = ...  # f) True or False
answer_B3_3g = ...  # g) a shape as a string, e.g. "(2, 3)"
answer_B3_3h = ...  # h) a whole number
answer_B3_3i = ...  # i) a shape as a string, e.g. "(2, 3)"

for letter, answer in zip("abcdefghi", [answer_B3_3a, answer_B3_3b, answer_B3_3c, answer_B3_3d, answer_B3_3e, answer_B3_3f, answer_B3_3g, answer_B3_3h, answer_B3_3i]):
    wb.check(f"B3.3{letter}", answer)

**Ex B3.4 — Encodage sinusoïdal : quelques valeurs et l'effet d'un décalage de position**

In [ ]:
# Ex B3.4: the values you found on paper
answer_B3_4a = ...  # a) a whole number
answer_B3_4b = ...  # b) a number, 3 decimals
answer_B3_4c = ...  # c) a number, 4 decimals
answer_B3_4d = ...  # d) a number, 4 decimals
answer_B3_4e = ...  # e) a number, 4 decimals
answer_B3_4f = ...  # f) a number, 4 decimals
answer_B3_4g = ...  # g) a number, 4 decimals
answer_B3_4h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B3_4a, answer_B3_4b, answer_B3_4c, answer_B3_4d, answer_B3_4e, answer_B3_4f, answer_B3_4g, answer_B3_4h]):
    wb.check(f"B3.4{letter}", answer)

**Ex B3.5 — Compter les paramètres d'un GPT et retrouver les 124 millions de GPT-2 small**

In [ ]:
# Ex B3.5: the values you found on paper
answer_B3_5a = ...  # a) a whole number
answer_B3_5b = ...  # b) a whole number
answer_B3_5c = ...  # c) a whole number
answer_B3_5d = ...  # d) a whole number
answer_B3_5e = ...  # e) a whole number
answer_B3_5f = ...  # f) a whole number
answer_B3_5g = ...  # g) a whole number
answer_B3_5h = ...  # h) a number, 1 decimal
answer_B3_5i = ...  # i) True or False
answer_B3_5j = ...  # j) a whole number

for letter, answer in zip("abcdefghij", [answer_B3_5a, answer_B3_5b, answer_B3_5c, answer_B3_5d, answer_B3_5e, answer_B3_5f, answer_B3_5g, answer_B3_5h, answer_B3_5i, answer_B3_5j]):
    wb.check(f"B3.5{letter}", answer)

## Partie A · Ta librairie NumPy : attention.py

Tu écris dans `mylearn/attention.py` les briques d'un Transformer, en NumPy, chacune comparée à PyTorch : les masques et l'attention par produit scalaire mise à l'échelle (B3.12), les têtes (B3.14), les encodages de position (B3.15), puis un bloc complet (B3.16, ★★★★). Entre deux, tu prédis ce que fait la division par $\sqrt{d_k}$ quand une clé copie la requête (B3.13), et tu répares un petit modèle de langue dont trois masques mentent (B3.17). `positional_encoding` resservira en B5, pour l'encodage du pas de temps d'un modèle de diffusion.

**Contrat commun de `attention.py`** (les docstrings, plus ces précisions que vérifient les tests) :
- conventions : couches denses comme partout dans `mylearn`, `W` de forme `(n_in, n_out)` et $z = xW + b$ ; masques booléens comme `F.scaled_dot_product_attention`, `True` = la requête **peut** regarder la clé (l'inverse de `nn.MultiheadAttention`) ;
- tes fonctions ne modifient jamais leurs entrées, ni les tableaux ni `params` ; les entrées peuvent être des listes (`lengths`, `q`, `k`, `v`, `mask`, `positions`, le `x` de `split_heads`) ;
- les masques rendus sont de dtype `bool` ; une entrée invalide lève `ValueError`, jamais `KeyError` ;
- les valeurs sont comparées à rtol = atol = $10^{-6}$ (un calcul en float32 passe ; $10^{-5}$ contre Hugging Face), les masques et les têtes exactement ;
- tu réutilises tes fonctions des chapitres précédents : `softmax` et `gelu` (`nn/activations.py`, ch. 17), `layer_norm` (`nn/regularization.py`, ch. 20). Le stub les importe. Si l'un de ces fichiers manque dans ta librairie, la référence le remplace (une ligne ℹ️ le dit) ; s'il existe mais que la fonction n'est pas écrite, les tests qui en dépendent affichent ⏳ en nommant son fichier et son exercice.

La cellule suivante charge les outils de la partie : `TEST_ATT` (le fichier des tests), `DATA` (Holmes en identifiants de caractères, le découpage figé de `wb.text.train_val("holmes")`, que tu as découpé en caractères au ch. 22), `causal_np` (le masque causal des vérifications, écrit sans ton code), `params_from_mha` et `block_params_from_torch` (les poids d'un module PyTorch dans ta convention).

In [ ]:
# Tools for the notebook exercises of part A (your NumPy library attention.py) and of the bug hunt B3.17
import copy
import math
import time
from types import MappingProxyType

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from wb import gpt as wbgpt
from wb.nbtools import StopCheck, checked, filled, fr, pair, refuses, returned, run_pytest_group, same, verdict, written

TEST_ATT = "tests/test_chb3_attention.py"      # the tests of your library attention.py
DATA = wbgpt.char_data("holmes")              # Holmes in character ids: the frozen split of wb.text.train_val


def causal_np(L, S=None):
    """The causal mask of the checks, aligned bottom-right (True = may attend), written once without your code."""
    S = L if S is None else S
    return np.tril(np.ones((L, S), dtype=bool), k=S - L)


def params_from_mha(mha):
    """The parameters of an nn.MultiheadAttention in the convention of multi_head_attention (z = x @ W + b,
    float64): PyTorch stores W_q, W_k and W_v transposed and stacked in in_proj_weight, (3D, D)."""
    D = mha.embed_dim
    W = mha.in_proj_weight.detach().double().numpy()
    b = mha.in_proj_bias.detach().double().numpy()
    return {"W_q": W[:D].T.copy(), "W_k": W[D:2 * D].T.copy(), "W_v": W[2 * D:].T.copy(),
            "W_o": mha.out_proj.weight.detach().double().numpy().T.copy(),
            "b_q": b[:D].copy(), "b_k": b[D:2 * D].copy(), "b_v": b[2 * D:].copy(),
            "b_o": mha.out_proj.bias.detach().double().numpy().copy()}


def block_params_from_torch(layer):
    """The parameters of an nn.TransformerEncoderLayer in the convention of transformer_block (float64)."""
    t = lambda w: w.detach().double().numpy().T.copy()        # noqa: E731  a dense layer: W = weight.T
    v = lambda b: b.detach().double().numpy().copy()          # noqa: E731
    return params_from_mha(layer.self_attn) | {
        "ln1_gamma": v(layer.norm1.weight), "ln1_beta": v(layer.norm1.bias),
        "ln2_gamma": v(layer.norm2.weight), "ln2_beta": v(layer.norm2.bias),
        "W_1": t(layer.linear1.weight), "b_1": v(layer.linear1.bias),
        "W_2": t(layer.linear2.weight), "b_2": v(layer.linear2.bias)}


print(f"Holmes : {len(DATA.train_text):_} caractères d'entraînement, {len(DATA.val_text):_} de validation, "
      f"{len(DATA.itos)} caractères distincts.".replace("_", " "))

### Ex B3.12 — causal_mask, padding_mask et scaled_dot_product_attention 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire les deux masques et l'attention par produit scalaire mise à l'échelle de ta librairie, comparés à PyTorch.  
**Prérequis :** ✏️ B3.2 · fiche §203.2 (« La formule »), §203.3 (« Le masque causal », « Le masque de padding », « Le masque causal quand il y a moins de requêtes que de clés ») · **mylearn :** `attention.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/attention.py` (créé par `python tools/start_chapter.py B3`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb3_attention.py -q` les lance tous dans un terminal.

**Contrat**
- `causal_mask(q_len, kv_len=None)` : le masque `(L, S)` de booléens $M_{ij} = (j \le i + S - L)$, aligné **en bas à droite** : la dernière requête voit toutes les clés (cas du KV cache, où les $S - L$ premières clés sont en cache) ; il coïncide avec `is_causal=True` de `F.scaled_dot_product_attention` quand $L = S$, pas quand $L < S$. `ValueError` si `q_len < 1` ou `kv_len < q_len`.
- `padding_mask(lengths, max_len)` : le masque `(N, S)` de booléens, `True` sur les vrais tokens, `False` sur le padding ; `ValueError` pour une longueur < 1 ou > `max_len`.
- `scaled_dot_product_attention(q, k, v, mask=None, scale=None)` rend le couple `(output, weights)` :
  - $\operatorname{softmax}(\mathbf{q}\mathbf{k}^\top \cdot \text{scale} + \mathbf{M})\,\mathbf{v}$ sur les deux derniers axes ; les axes de tête (batch, têtes) sont gardés et diffusés, et `mask` se diffuse sur `(..., L, S)` : `(L, S)`, `(N, 1, 1, S)`… ;
  - une clé masquée reçoit un poids de 0 **exactement**, et chaque ligne de poids somme à 1 ;
  - `scale` : seul `None` donne $1/\sqrt{d_k}$ ; `scale=0.0` est une échelle comme une autre (des scores nuls) ;
  - `ValueError` si les formes sont incompatibles, ou si une ligne du masque n'autorise **aucune** clé, y compris quand cette ligne vient d'un masque 4-D `(N, 1, 1, S)` (une séquence entièrement masquée). `F.scaled_dot_product_attention` rendrait une ligne de zéros et `nn.MultiheadAttention` des `nan` : ta librairie refuse.

La première cellule de vérification essaie tes masques sur des cas visibles (dont ceux des docstrings) et contre `torch.tril`, refait l'attention à la main de la fiche (§203.2 : une requête, quatre clés), compare ton attention à `F.scaled_dot_product_attention` sur un batch à têtes avec un masque de padding `(2, 1, 1, 7)` et avec le masque causal, essaie `scale=0.0` et quatre refus. La seconde lance les 55 tests de l'exercice (`-k "test_causal_mask_ or test_padding_mask_ or test_scaled_dot_product_attention_"`).

Dans tes notes : avec `causal_mask(2, 5)`, quelle requête est le token de quelle position, et que donnerait `is_causal=True` à la place ? Pourquoi ta fonction refuse-t-elle une ligne sans aucune clé autorisée, plutôt que de rendre des zéros comme SDPA ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B3")   # reload your saved file
with checked("B3.12"):
    A = mylearn.attention
    m_12 = A.causal_mask(3)
    if returned("B3.12", "causal_mask", m_12):
        arr_12 = np.asarray(m_12)
        verdict("B3.12", arr_12.dtype == bool and np.array_equal(arr_12, causal_np(3)),
                "causal_mask(3) : le triangle inférieur, diagonale comprise, en booléens.",
                f"causal_mask(3) doit rendre le triangle inférieur, diagonale comprise, de dtype bool ; reçu "
                f"{arr_12.astype(int).tolist()} de dtype {arr_12.dtype}.")
    for L_12, S_12 in ((2, 4), (5, 5), (3, 8)):
        want_12 = torch.tril(torch.ones(L_12, S_12, dtype=torch.bool), diagonal=S_12 - L_12).numpy()
        got_12 = np.asarray(A.causal_mask(L_12, S_12))
        verdict("B3.12", got_12.shape == want_12.shape and np.array_equal(got_12, want_12),
                f"causal_mask({L_12}, {S_12}) = torch.tril(…, diagonal={S_12 - L_12}) : aligné en bas à droite.",
                f"causal_mask({L_12}, {S_12}) doit valoir torch.tril(torch.ones({L_12}, {S_12}), diagonal={S_12 - L_12}) "
                f"(la dernière requête voit toutes les clés) ; reçu {got_12.astype(int).tolist()}.")
    refuses("B3.12", "causal_mask(3, 2) (moins de clés que de requêtes)", lambda: A.causal_mask(3, 2))
    pad_12 = np.asarray(A.padding_mask([2, 3], 4))
    verdict("B3.12", pad_12.dtype == bool and np.array_equal(pad_12, [[True, True, False, False], [True, True, True, False]]),
            "padding_mask([2, 3], 4) : True sur les vrais tokens, False sur le padding.",
            f"padding_mask([2, 3], 4) doit rendre [[T, T, F, F], [T, T, T, F]] de dtype bool ; reçu {pad_12.astype(int).tolist()}.")
    refuses("B3.12", "padding_mask([0, 2], 4) (une longueur nulle)", lambda: A.padding_mask([0, 2], 4))
    refuses("B3.12", "padding_mask([5], 4) (plus long que max_len)", lambda: A.padding_mask([5], 4))

    # the attention by hand of the fiche (§203.2): one query, four keys, d_k = 4
    q_f = [[1.0, 0.0, 1.0, 0.0]]                                         # a list: accepted
    k_f = np.array([[1.0, 1, 0, 0], [0, 1, 1, 0], [2, 0, 2, 0], [0, 0, 0, 2]])
    v_f = np.array([[1.0, 0], [0, 1], [2, 2], [-1, 1]])
    k_copy_12 = k_f.copy()
    scores_f = np.array(q_f) @ k_f.T / 2.0
    w_want_12 = np.exp(scores_f) / np.exp(scores_f).sum()
    out_f, w_f = pair("B3.12", "scaled_dot_product_attention", A.scaled_dot_product_attention(q_f, k_f, v_f),
                      names=("output", "weights"))
    same("B3.12", "poids de l'exemple de la fiche (§203.2)", w_f, w_want_12, rtol=1e-6, atol=1e-6)
    same("B3.12", "sortie de l'exemple de la fiche", out_f, w_want_12 @ v_f, rtol=1e-6, atol=1e-6)
    verdict("B3.12", np.array_equal(k_f, k_copy_12), "les clés reçues n'ont pas changé.",
            "scaled_dot_product_attention a modifié les clés reçues.")
    _, w_c = A.scaled_dot_product_attention(q_f, k_f, v_f, mask=[[True, True, True, False]])
    verdict("B3.12", np.asarray(w_c)[0, 3] == 0 and np.isclose(np.asarray(w_c).sum(), 1),
            "une clé masquée reçoit un poids de 0 exactement, et la ligne somme encore à 1.",
            f"avec la clé 3 masquée, attendu un poids de 0 exactement et une ligne qui somme à 1 ; reçu {np.round(w_c, 4).tolist()}.")

    # a batch with heads and a padding mask (N, 1, 1, S), against F.scaled_dot_product_attention (float64)
    rng_12 = np.random.default_rng(12)
    q_12, k_12, v_12 = rng_12.normal(size=(2, 3, 5, 8)), rng_12.normal(size=(2, 3, 7, 8)), rng_12.normal(size=(2, 3, 7, 4))
    keys_12 = (np.arange(7) < np.array([7, 4])[:, None])[:, None, None, :]          # sequence 1 has 4 real keys
    q_copy_12 = q_12.copy()
    want_12 = F.scaled_dot_product_attention(*(torch.from_numpy(a) for a in (q_12, k_12, v_12)),
                                             attn_mask=torch.from_numpy(keys_12)).numpy()
    out_12, w_12 = pair("B3.12", "scaled_dot_product_attention", A.scaled_dot_product_attention(q_12, k_12, v_12, mask=keys_12),
                        names=("output", "weights"))
    same("B3.12", "sortie (2, 3, 5, 4) avec un masque de padding (2, 1, 1, 7), contre F.scaled_dot_product_attention",
         out_12, want_12, rtol=1e-6, atol=1e-6)
    if not np.array_equal(q_12, q_copy_12):
        print("❌ Ex B3.12 : scaled_dot_product_attention a modifié les requêtes reçues (la suite part de requêtes neuves).")
        q_12 = q_copy_12.copy()
    out_c12, _ = A.scaled_dot_product_attention(q_12[..., :5, :], k_12[..., :5, :], v_12[..., :5, :], mask=causal_np(5))
    want_c12 = F.scaled_dot_product_attention(*(torch.from_numpy(a[..., :5, :]) for a in (q_12, k_12, v_12)), is_causal=True).numpy()
    same("B3.12", "masque causal (5, 5) contre is_causal=True", out_c12, want_c12, rtol=1e-6, atol=1e-6)
    _, w0_12 = A.scaled_dot_product_attention(q_12, k_12, v_12, scale=0.0)
    verdict("B3.12", np.allclose(w0_12, 1 / 7),
            "scale=0.0 : des scores nuls, donc des poids uniformes (seul None donne 1/√d_k).",
            "scale=0.0 doit s'appliquer tel quel (des scores nuls, des poids uniformes de 1/7) : seul None donne 1/√d_k.")
    empty_12 = keys_12.copy()
    empty_12[1] = False                                                  # sequence 1 entirely masked
    refuses("B3.12", "un masque (2, 1, 1, 7) qui n'autorise aucune clé à la séquence 1",
            lambda: A.scaled_dot_product_attention(q_12, k_12, v_12, mask=empty_12))

In [ ]:
if written(lambda: mylearn.attention.causal_mask(2)):
    run_pytest_group(TEST_ATT, "test_causal_mask_ or test_padding_mask_ or test_scaled_dot_product_attention_")   # 55 tests
else:
    print("⏳ Ex B3.12 : les tests attendent tes fonctions.")

### Ex B3.13 — Prédire la carte d'attention quand une clé copie la requête, puis quand on retire la division par √d_k 🔮 ★ ⏱️ 10 min
**Objectif :** prédire, en ordre de grandeur, combien de clés aléatoires il faut pour qu'une clé égale à la requête ne prenne plus la moitié du poids, avec et sans la division par $\sqrt{d_k}$.  
**Prérequis :** Ex B3.12 · ∂ B3.6 · fiche §203.2 (« Pourquoi diviser par $\sqrt{d_k}$ ») · **Parcours :** C

Une requête $\mathbf{q}$ de dimension $d_k = 64$, de composantes indépendantes tirées selon $\mathcal{N}(0;\ 1)$. Parmi ses $S$ clés, l'une est $\mathbf{q}$ elle-même : une clé qui **copie** la requête ; les $S - 1$ autres sont tirées comme $\mathbf{q}$, indépendamment. La cellule suivante donne `copied_weight_13(S, scaled)`, qui calcule pour 200 requêtes le poids d'attention de la clé copiée parmi $S$ clés, avec (`scaled=True`) ou sans (`scaled=False`) la division par $\sqrt{d_k}$.

L'expérience trace d'abord deux cartes d'attention avec ta `scaled_dot_product_attention` (6 requêtes et 32 clés, dont les 6 premières copient les requêtes), avec puis sans la division ; puis le poids **médian** de la clé copiée, sur les 200 requêtes, pour $S$ = 10, 30, 100, 300, …, 100 000. **Avant d'exécuter**, écris tes deux prédictions :
- a) `exponent_13` : **avec** la division, l'entier $k$, de 0 à 5, tel que le poids médian de la clé copiée passe sous 1/2 entre $10^k$ et $10^{k+1}$ clés (par exemple `1` si la bascule a lieu entre 10 et 100 clés ; 0 si elle a lieu avant 10 clés, 5 si elle n'a pas encore eu lieu à 100 000) ;
- b) `keeps_half_13` : **sans** la division, avec 100 000 clés, le poids médian de la clé copiée est-il encore au-dessus de 1/2 ? `True` ou `False`.

La cellule d'expérience n'affiche rien tant que tes deux prédictions ne sont pas écrites ; elle les vérifie à la fin. La courbe ne dépend pas de ton code ; les deux cartes, si (B3.12).

In [ ]:
D_13, QUERIES_13 = 64, 200                                           # d_k = 64, 200 queries
KEYS_13 = (10, 30, 100, 300, 1_000, 3_000, 10_000, 30_000, 100_000)       # numbers of keys of the experiment


def copied_weight_13(n_keys, scaled=True, d_k=D_13, n_queries=QUERIES_13, seed=13, chunk=10_000):
    """For n_queries Gaussian queries q of dimension d_k (components N(0, 1)): the attention weight of the key equal
    to q among n_keys keys (q itself, then n_keys - 1 Gaussian keys), with or without the division by sqrt(d_k).
    The softmax over the keys is summed by blocks of `chunk` keys (log-sum-exp), so that 100 000 keys fit in memory."""
    rng = np.random.default_rng(seed)
    q = rng.normal(size=(n_queries, d_k))
    scale = 1 / math.sqrt(d_k) if scaled else 1.0
    own = (q * q).sum(axis=1) * scale                                   # the score of the copied key
    others = np.full(n_queries, -np.inf)                                # log of the sum of exp(score) of the others
    left = n_keys - 1
    while left > 0:
        k = rng.normal(size=(min(chunk, left), d_k))
        s = q @ k.T * scale
        top = s.max(axis=1)
        others = np.logaddexp(others, top + np.log(np.exp(s - top[:, None]).sum(axis=1)))
        left -= len(k)
    return 1 / (1 + np.exp(others - own))                               # exp(own) / (exp(own) + the others)

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
exponent_13 = ...      # a) WITH the division by sqrt(d_k): the integer k, 0 to 5, such that the median weight of the
                       #    copied key falls under 1/2 between 10**k and 10**(k + 1) keys
keeps_half_13 = ...    # b) WITHOUT the division: with 100 000 keys, is the median weight of the copied key still
                       #    above 1/2? True or False

In [ ]:
if any(answer is ... or answer is None for answer in [exponent_13, keeps_half_13]):
    print("⏳ Ex B3.13 : écris d'abord tes deux prédictions dans la cellule précédente.")
else:
    # 1. the attention map: 6 queries; keys 0 to 5 copy them, keys 6 to 31 are random (your scaled_dot_product_attention)
    rng_13 = np.random.default_rng(130)
    q_13 = rng_13.normal(size=(6, D_13))
    k_13 = np.vstack([q_13, rng_13.normal(size=(26, D_13))])
    maps_13 = None
    with checked("B3.13"):
        maps_13 = {"avec la division par √d_k": np.asarray(mylearn.attention.scaled_dot_product_attention(q_13, k_13, k_13)[1]),
                   "sans la division (scale = 1)": np.asarray(mylearn.attention.scaled_dot_product_attention(q_13, k_13, k_13, scale=1.0)[1])}
    if maps_13 is not None:
        fig, axes = plt.subplots(1, 2, figsize=(12, 2.8))
        for ax, (title, w) in zip(axes, maps_13.items()):
            im = ax.imshow(w, cmap="viridis", vmin=0, vmax=1, aspect="auto")
            ax.set_title(title)
            ax.set_xlabel("clé (0 à 5 : copies des requêtes)")
            ax.set_ylabel("requête")
        fig.colorbar(im, ax=axes, shrink=0.8)
        plt.show()
        print("poids de la clé copiée, requête par requête :",
              {title: np.round(np.diag(w[:, :6]), 3).tolist() for title, w in maps_13.items()})
    else:
        print("⏳ Ex B3.13 : la carte attend ta scaled_dot_product_attention (B3.12) ; la courbe ci-dessous n'en a pas besoin.")

    # 2. the median weight of the copied key over 200 queries, from 10 to 100 000 keys
    medians_13 = {scaled: [float(np.median(copied_weight_13(n, scaled))) for n in KEYS_13] for scaled in (True, False)}
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.semilogx(KEYS_13, medians_13[True], "o-", label="avec la division par √d_k")
    ax.semilogx(KEYS_13, medians_13[False], "s--", label="sans la division")
    ax.axhline(0.5, color="gray", lw=0.8)
    ax.set_xlabel("nombre de clés (la clé copiée et des clés aléatoires)")
    ax.set_ylabel("poids médian de la clé copiée")
    ax.set_ylim(-0.03, 1.03)
    ax.legend()
    plt.show()
    for n_13, with_13, without_13 in zip(KEYS_13, medians_13[True], medians_13[False]):
        print(f"{n_13:>7_} clés : {fr(with_13, 3)} avec la division, {fr(without_13, 6)} sans".replace("_", " "))
    wb.check("B3.13a", exponent_13)
    wb.check("B3.13b", keeps_half_13)

Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? Relie la courbe « avec la division » à ∂ B3.6 : quel est le score typique de la clé copiée, celui des autres clés, et que compare la softmax ? Refais la courbe avec `d_k = 16`, puis `d_k = 256` (`copied_weight_13(n, True, d_k=…)`) : où se place la bascule, et que corrige exactement la division par $\sqrt{d_k}$ ?

### Ex B3.14 — split_heads, merge_heads et multi_head_attention 🔨 ★★★ ⏱️ 45 min
**Objectif :** découper en têtes, recoller, puis assembler une attention multi-têtes complète, validée contre `nn.MultiheadAttention`.  
**Prérequis :** Ex B3.12 · ✏️ B3.3 · fiche §203.4 (« Une seule matrice pour toutes les têtes », piège du reshape direct) · **mylearn :** `attention.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/attention.py` (créé par `python tools/start_chapter.py B3`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb3_attention.py -q` les lance tous dans un terminal.

**Contrat**
- `split_heads(x, n_heads)` : `(N, T, D)` → `(N, H, T, D // H)` ; la tête $h$ reçoit les features $h\,d_h$ à $(h + 1)\,d_h - 1$ de chaque token ($d_h = D / H$). `ValueError` si `x` n'est pas 3-D ou si $H$ ne divise pas $D$.
- `merge_heads(x)` : l'inverse, `(N, H, T, d_h)` → `(N, T, H d_h)`, la tête 0 d'abord ; `merge_heads(split_heads(x, H))` redonne `x`. `ValueError` si `x` n'est pas 4-D.
- `multi_head_attention(query, key, value, params, n_heads, mask=None)` rend le couple `(output, weights)`, de formes `(N, L, D)` et `(N, H, L, S)` :
  - $\mathbf{Q} = \text{query}\,\mathbf{W}_q + \mathbf{b}_q$ (de même $\mathbf{K}$ avec `key` et $\mathbf{V}$ avec `value`), le découpage en têtes, ta `scaled_dot_product_attention` (B3.12) dans chaque tête avec son échelle par défaut, le recollage, puis la projection de sortie $\mathbf{W}_o$, $\mathbf{b}_o$ ; l'auto-attention est `query = key = value = x` ;
  - `params` : `W_q`, `W_k`, `W_v`, `W_o` de forme `(D, D)` et `b_q`, `b_k`, `b_v`, `b_o` de forme `(D,)`, dans un mapping quelconque (aussi en lecture seule, `MappingProxyType`) ; ses autres clés sont ignorées (les paramètres d'un bloc entier passent) ; un paramètre manquant lève `ValueError`, pas `KeyError` ;
  - `mask` (`True` = autorisé) de forme `(L, S)`, `(N, 1, 1, S)`, `(N, 1, L, S)` ou `(N, H, L, S)` ; les tests prennent aussi de l'attention croisée, où $L \ne S$.

La première cellule de vérification essaie tes deux fonctions de têtes sur l'exemple de la docstring, contre `view(N, T, H, d_h).transpose(1, 2)` de PyTorch, puis l'aller-retour et deux refus. Elle compare ensuite ton attention multi-têtes à `nn.MultiheadAttention` aux mêmes poids (la fonction `params_from_mha` de la cellule des outils montre la conversion : PyTorch range $\mathbf{W}_q^\top$, $\mathbf{W}_k^\top$, $\mathbf{W}_v^\top$ empilées dans `in_proj_weight`, et attend le masque inversé, `~mask`), en auto-attention causale puis en attention croisée avec un masque de padding, et essaie un paramètre manquant. La seconde lance les 48 tests de l'exercice (`-k "test_split_heads_ or test_merge_heads_ or test_multi_head_attention_"`).

Dans tes notes : un reshape direct `(N, T, D)` → `(N, H, T, d_h)`, sans échange d'axes, donne la bonne forme : quelles features reçoit alors chaque tête (fiche §203.4) ? Pourquoi `params_from_mha` transpose-t-elle chaque bloc de `in_proj_weight` ? Que changerait une échelle $1/\sqrt{D}$ au lieu de $1/\sqrt{d_h}$ ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B3")   # reload your saved file
with checked("B3.14"):
    A = mylearn.attention
    x_14 = np.arange(12.0).reshape(1, 2, 6)
    h_14 = A.split_heads(x_14, 3)
    if returned("B3.14", "split_heads", h_14):
        verdict("B3.14", np.shape(h_14) == (1, 3, 2, 2) and np.array_equal(np.asarray(h_14)[0, 1], [[2.0, 3.0], [8.0, 9.0]]),
                "split_heads(arange(12).reshape(1, 2, 6), 3) : forme (1, 3, 2, 2), la tête 1 a les features 2 et 3.",
                f"split_heads(x, 3) doit rendre la forme (1, 3, 2, 2) et la tête 1 [[2, 3], [8, 9]] ; reçu la forme "
                f"{np.shape(h_14)} et la tête 1 {np.asarray(h_14)[0, 1].tolist() if np.ndim(h_14) == 4 else '?'}.")
    rng_14 = np.random.default_rng(14)
    x_14 = rng_14.normal(size=(2, 5, 12))
    want_14 = torch.from_numpy(x_14).view(2, 5, 4, 3).transpose(1, 2).numpy()
    same("B3.14", "split_heads(x (2, 5, 12), 4) contre view(2, 5, 4, 3).transpose(1, 2)", A.split_heads(x_14, 4), want_14,
         rtol=0, atol=0)
    same("B3.14", "merge_heads(split_heads(x, 4)), qui doit redonner x", A.merge_heads(A.split_heads(x_14, 4)), x_14, rtol=0, atol=0)
    refuses("B3.14", "split_heads d'un x 2-D", lambda: A.split_heads(np.ones((5, 12)), 4))
    refuses("B3.14", "split_heads avec 5 têtes pour D = 12", lambda: A.split_heads(x_14, 5))

    # multi_head_attention against nn.MultiheadAttention with the same weights (float64)
    torch.manual_seed(14)
    D_14, H_14 = 12, 3
    mha_14 = nn.MultiheadAttention(D_14, H_14, batch_first=True).double()
    params_14 = params_from_mha(mha_14) | {"ln1_gamma": np.ones(D_14)}         # an extra key: ignored
    saved_14 = {name: p.copy() for name, p in params_14.items()}
    xt_14 = torch.from_numpy(x_14)
    with torch.no_grad():
        want_o, want_w = mha_14(xt_14, xt_14, xt_14, attn_mask=torch.from_numpy(~causal_np(5)), average_attn_weights=False)
    out_14, w_14 = pair("B3.14", "multi_head_attention", A.multi_head_attention(x_14, x_14, x_14, MappingProxyType(params_14), H_14,
                                                                             mask=causal_np(5)), names=("output", "weights"))
    same("B3.14", "auto-attention causale (2, 5, 12), 3 têtes, contre nn.MultiheadAttention (masque ~mask)", out_14, want_o.numpy(),
         rtol=1e-6, atol=1e-6)
    same("B3.14", "poids par tête (2, 3, 5, 5)", w_14, want_w.numpy(), rtol=1e-6, atol=1e-6)
    enc_14 = rng_14.normal(size=(2, 9, D_14))
    keys_14 = np.arange(9) < np.array([9, 6])[:, None]                        # sequence 1: 6 real keys out of 9
    with torch.no_grad():
        want_x, _ = mha_14(xt_14, torch.from_numpy(enc_14), torch.from_numpy(enc_14), key_padding_mask=torch.from_numpy(~keys_14))
    out_x, w_x = pair("B3.14", "multi_head_attention", A.multi_head_attention(x_14, enc_14, enc_14, params_14, H_14,
                                                                           mask=keys_14[:, None, None, :]), names=("output", "weights"))
    same("B3.14", "attention croisée (5 requêtes, 9 clés) avec un masque de padding (2, 1, 1, 9)", out_x, want_x.numpy(),
         rtol=1e-6, atol=1e-6)
    verdict("B3.14", all(np.array_equal(params_14[name], saved_14[name]) for name in saved_14),
            "les paramètres reçus n'ont pas changé.", "multi_head_attention a modifié les paramètres reçus.")
    no_wo_14 = {name: p for name, p in params_14.items() if name != "W_o"}
    refuses("B3.14", "multi_head_attention sans W_o (ValueError, pas KeyError)",
            lambda: A.multi_head_attention(x_14, x_14, x_14, no_wo_14, H_14))

In [ ]:
if written(lambda: mylearn.attention.split_heads(np.ones((1, 2, 4)), 2)):
    run_pytest_group(TEST_ATT, "test_split_heads_ or test_merge_heads_ or test_multi_head_attention_")   # 48 tests
else:
    print("⏳ Ex B3.14 : les tests attendent tes fonctions.")

### Ex B3.15 — positional_encoding et apply_rope 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire l'encodage sinusoïdal et RoPE, et vérifier leurs propriétés et la convention de Hugging Face.  
**Prérequis :** ✏️ B3.4 · ∂ B3.7 · fiche §203.5 (« L'encodage sinusoïdal (2017) », « RoPE : tourner les requêtes et les clés ») · **mylearn :** `attention.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/attention.py` (créé par `python tools/start_chapter.py B3`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb3_attention.py -q` les lance tous dans un terminal.

**Contrat**
- `positional_encoding(seq_len, d_model, base=10000.0)` : le tableau `(T, D)` en float64 de $PE_{p, 2i} = \sin(p / \text{base}^{2i/D})$ et $PE_{p, 2i+1} = \cos(p / \text{base}^{2i/D})$, pour $i = 0 \dots D/2 - 1$. `base = 1` est accepté ; `ValueError` si `d_model` est impair, ou si `seq_len`, `d_model` ou `base` est < 1. Les tests vont jusqu'à $T = 1\,000$ : B5 s'en resservira.
- `apply_rope(x, positions=None, base=10000.0)` : `x` de forme `(..., T, d)`, $d$ pair, une requête ou une clé par position ; la convention **« rotate half »** de Hugging Face (Llama) : la dimension $i$ est appariée avec $i + d/2$ (pour $i < d/2$), pas avec $i + 1$ comme dans l'article de RoFormer, et la paire $(x_1, x_2)$ tourne de l'angle $\theta = \text{position} \cdot \text{base}^{-2i/d}$ :
  $(x_1 \cos\theta - x_2 \sin\theta,\ x_2 \cos\theta + x_1 \sin\theta)$ ;
  - `positions` : exactement $T$ positions (`np.arange(T)` si `None`) ; une seule position pour $T$ vecteurs est refusée, même si NumPy la diffuserait ;
  - rend un tableau de la forme de `x` ; `ValueError` si $d$ est impair ou si `positions` n'a pas la longueur $T$ ;
  - comparé à `LlamaRotaryEmbedding` et `apply_rotary_pos_emb` de transformers, à $10^{-5}$ (ils calculent les angles en float32).

La première cellule de vérification essaie ton encodage sur l'exemple de la docstring, sur 1 000 positions et sur la propriété du décalage (§203.5 : décaler de 7 positions applique la même rotation à chaque paire (sin, cos)), puis RoPE sur l'exemple de la docstring, la conservation des normes, la propriété relative (le produit scalaire ne dépend que de $m - n$, ∂ B3.7) et contre `LlamaRotaryEmbedding` ; enfin quatre refus. La seconde lance les 40 tests de l'exercice (`-k "test_positional_encoding_ or test_apply_rope_"`).

Dans tes notes : pourquoi RoPE tourne-t-il les requêtes et les clés, mais pas les valeurs ? Le produit scalaire d'une requête et d'une clé change-t-il quand on décale toutes les positions de 1 000 (`positions = np.arange(T) + 1000`) ? Mesure-le. Qu'apporte la convention « rotate half » par rapport aux paires $(2i, 2i + 1)$, et que perd-on à confondre les deux ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B3")   # reload your saved file
with checked("B3.15"):
    A = mylearn.attention
    pos_15 = np.arange(3)[:, None] / 10000.0 ** (np.arange(0, 4, 2) / 4)       # the formula of the docstring
    want_15 = np.stack([np.sin(pos_15), np.cos(pos_15)], axis=-1).reshape(3, 4)
    same("B3.15", "positional_encoding(3, 4)", A.positional_encoding(3, 4), want_15, rtol=1e-6, atol=1e-6)
    pe_15 = np.asarray(A.positional_encoding(1000, 64))
    verdict("B3.15", pe_15.shape == (1000, 64) and pe_15.dtype == np.float64 and np.abs(pe_15).max() <= 1,
            "positional_encoding(1000, 64) : forme (1000, 64), float64, valeurs dans [−1, 1].",
            f"positional_encoding(1000, 64) : attendu un tableau float64 (1000, 64) dans [−1, 1] ; reçu {pe_15.dtype} {pe_15.shape}.")
    omega_15 = 10000.0 ** (-np.arange(32) / 32)                                  # one frequency per (sin, cos) pair
    c_15, s_15 = np.cos(7 * omega_15), np.sin(7 * omega_15)
    sin_15, cos_15 = pe_15[:-7, 0::2], pe_15[:-7, 1::2]
    shifted_15 = np.stack([sin_15 * c_15 + cos_15 * s_15, cos_15 * c_15 - sin_15 * s_15], axis=-1).reshape(993, 64)
    same("B3.15", "un décalage de 7 positions = une rotation fixe de chaque paire (sin, cos) (§203.5)", pe_15[7:], shifted_15,
         rtol=1e-6, atol=1e-6)
    refuses("B3.15", "positional_encoding(4, 5) (d_model impair)", lambda: A.positional_encoding(4, 5))
    refuses("B3.15", "positional_encoding(0, 4)", lambda: A.positional_encoding(0, 4))

    ones_15 = np.ones((3, 4))
    half_15 = 10000.0 ** (-2 * np.arange(2) / 4)                                  # theta = position * base**(-2i/d)
    z_15 = (1 + 1j) * np.exp(1j * np.arange(3)[:, None] * half_15)                # pair (x_i, x_{i+d/2}) as x_i + i x_{i+d/2}
    same("B3.15", "apply_rope(np.ones((3, 4))) (paires (i, i + d/2))", A.apply_rope(ones_15),
         np.concatenate([z_15.real, z_15.imag], axis=1), rtol=1e-6, atol=1e-6)
    rng_15 = np.random.default_rng(15)
    x_15 = rng_15.normal(size=(2, 3, 10, 8))
    same("B3.15", "normes de apply_rope(x (2, 3, 10, 8)), égales à celles de x", np.linalg.norm(A.apply_rope(x_15), axis=-1),
         np.linalg.norm(x_15, axis=-1), rtol=1e-6, atol=1e-6)
    q_15, k_15 = rng_15.normal(size=(1, 8)), rng_15.normal(size=(1, 8))
    dots_15 = [(np.asarray(A.apply_rope(q_15, [m])) @ np.asarray(A.apply_rope(k_15, [n])).T).item()
               for m, n in ((3, 1), (12, 10), (102, 100))]
    verdict("B3.15", np.allclose(dots_15, dots_15[0], rtol=1e-6, atol=1e-6),
            "⟨RoPE(q, m), RoPE(k, n)⟩ ne dépend que de m − n : " + ", ".join(fr(d, 4) for d in dots_15) + " pour (3, 1), (12, 10), (102, 100).",
            "⟨RoPE(q, m), RoPE(k, n)⟩ doit être le même pour (3, 1), (12, 10) et (102, 100) (m − n = 2) ; reçu "
            + ", ".join(fr(d, 4) for d in dots_15) + ".")
    try:
        from transformers.models.llama.configuration_llama import LlamaConfig
        from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding, apply_rotary_pos_emb
    except ImportError:
        print("⏳ Ex B3.15 : la comparaison à LlamaRotaryEmbedding attend la bibliothèque transformers (Colab).")
    else:
        config_15 = LlamaConfig(hidden_size=24, num_attention_heads=3, head_dim=8, max_position_embeddings=128,
                                rope_parameters={"rope_type": "default", "rope_theta": 10000.0})
        cos_hf, sin_hf = LlamaRotaryEmbedding(config_15)(torch.from_numpy(x_15), torch.arange(10)[None, :])
        want_hf, _ = apply_rotary_pos_emb(torch.from_numpy(x_15), torch.from_numpy(x_15), cos_hf, sin_hf)
        same("B3.15", "apply_rope(x) contre LlamaRotaryEmbedding + apply_rotary_pos_emb de transformers (angles en float32 : 1e-5)",
             A.apply_rope(x_15), want_hf.numpy(), rtol=1e-5, atol=1e-5)
    refuses("B3.15", "apply_rope(np.ones((3, 4)), positions=[0]) (une position pour trois vecteurs)",
            lambda: A.apply_rope(ones_15, positions=[0]))
    refuses("B3.15", "apply_rope(np.ones((3, 5))) (d impair)", lambda: A.apply_rope(np.ones((3, 5))))

In [ ]:
if written(lambda: mylearn.attention.positional_encoding(2, 2)):
    run_pytest_group(TEST_ATT, "test_positional_encoding_ or test_apply_rope_")   # 40 tests
else:
    print("⏳ Ex B3.15 : les tests attendent tes fonctions.")

### Ex B3.16 — transformer_block en NumPy, validé contre nn.TransformerEncoderLayer 🔨 ★★★★ ⏱️ 100 min
**Objectif :** assembler un bloc Transformer pré-LN complet en NumPy, et le valider au millionième contre PyTorch.  
**Prérequis :** Ex B3.14 · ch. 20 (layer norm, 20.17) · ch. 17 (GELU, 17.14) · fiche §203.6 (« Pré-LN ou post-LN », « Le bloc en équations, et sa vérification ») · **mylearn :** `attention.py` · **Parcours :** C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/attention.py` (créé par `python tools/start_chapter.py B3`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb3_attention.py -q` les lance tous dans un terminal.

**Contrat** de `transformer_block(x, params, n_heads, mask=None, eps=1e-5)`, le bloc pré-LN de GPT-2, sans dropout :
- $\mathbf{h} = \mathbf{x} + \operatorname{MHA}(\operatorname{LN}_1(\mathbf{x}))$ en auto-attention (ta `multi_head_attention`, avec `mask`), puis $\text{out} = \mathbf{h} + \operatorname{gelu}(\operatorname{LN}_2(\mathbf{h})\,\mathbf{W}_1 + \mathbf{b}_1)\,\mathbf{W}_2 + \mathbf{b}_2$ ;
- la GELU **exacte** (avec la fonction d'erreur), pas l'approximation tanh ; `eps` sert aux **deux** layer norms ;
- `params` : les clés de `multi_head_attention`, plus `ln1_gamma`, `ln1_beta`, `ln2_gamma`, `ln2_beta` de forme `(D,)`, `W_1` `(D, F)`, `b_1` `(F,)`, `W_2` `(F, D)` et `b_2` `(D,)` ; la largeur $F$ du MLP se lit dans `W_1` ; un paramètre manquant lève `ValueError` ;
- rend `(N, T, D)` ; `x` et `params` ne sont pas modifiés.

**Tes fonctions des ch. 17 et 20.** Le stub importe ta `gelu` (`nn/activations.py`, Ex 17.14) et ta `layer_norm` (`nn/regularization.py`, Ex 20.17) : réutilise-les. Si l'un de ces fichiers manque dans ta librairie, la référence le remplace (une ligne ℹ️ le dit). S'il existe mais que la fonction n'est pas encore écrite, la première cellule de vérification te le signale : écris-la (son exercice), ou recode la layer norm et la GELU dans `attention.py`, ce que les tests acceptent aussi. Les tests passent par tes fonctions si tu les réutilises : si l'un d'eux échoue, lance d'abord les tests de 17.14 et de 20.17 (le message de l'échec le dit).

La première cellule de vérification construit le même bloc en PyTorch, `nn.TransformerEncoderLayer(16, 4, 64, dropout=0.0, activation="gelu", batch_first=True, norm_first=True)` en float64, recopie ses poids dans ta convention (`block_params_from_torch` dans la cellule des outils : `W = weight.T` pour chaque couche dense), et compare ta sortie à la sienne avec le masque causal (passé à PyTorch en `~mask`), sans masque, puis avec `eps = 0.1` ; elle essaie enfin un paramètre manquant. La seconde lance les 25 tests de l'exercice (`-k "test_transformer_block_"`).

Dans tes notes : avec tous les poids à zéro (l'exemple de la docstring), que rend ton bloc, et que dit ce résultat des connexions résiduelles ? En quoi ton bloc changerait-il pour un post-LN (§203.6) ? Remplace ta GELU par l'approximation tanh : de combien ta sortie s'écarte-t-elle de celle de PyTorch, et le test l'aurait-il vu ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B3")   # reload your saved file
with checked("B3.16"):
    A = mylearn.attention
    for name_16, probe_16, where_16 in (
            ("gelu", lambda: mylearn.nn.activations.gelu(np.zeros(2)), "nn/activations.py, Ex 17.14"),
            ("layer_norm", lambda: mylearn.nn.regularization.layer_norm(np.zeros((1, 4)), np.ones(4), np.zeros(4)),
             "nn/regularization.py, Ex 20.17")):
        if not written(probe_16):
            print(f"ℹ️ Ex B3.16 : ta {name_16} ({where_16}) n'est pas encore écrite : écris-la, ou recode-la dans "
                  "transformer_block (l'énoncé le permet).")
    # the same block in PyTorch: nn.TransformerEncoderLayer, pre-LN, exact GELU, no dropout, in float64
    torch.manual_seed(16)
    D_16, H_16, F_16 = 16, 4, 64
    layer_16 = nn.TransformerEncoderLayer(D_16, H_16, F_16, dropout=0.0, activation="gelu", batch_first=True,
                                          norm_first=True).double().eval()
    params_16 = block_params_from_torch(layer_16)
    saved_16 = {name: p.copy() for name, p in params_16.items()}
    x_16 = np.random.default_rng(16).normal(size=(2, 7, D_16))
    x_copy_16 = x_16.copy()
    with torch.no_grad():
        want_c16 = layer_16(torch.from_numpy(x_16), src_mask=torch.from_numpy(~causal_np(7))).numpy()
        want_n16 = layer_16(torch.from_numpy(x_16)).numpy()
    same("B3.16", "transformer_block(x (2, 7, 16), 4 têtes, masque causal) contre nn.TransformerEncoderLayer(norm_first=True)",
         A.transformer_block(x_16, params_16, H_16, mask=causal_np(7)), want_c16, rtol=1e-6, atol=1e-6)
    same("B3.16", "le même bloc sans masque (un encodeur)", A.transformer_block(x_16, params_16, H_16), want_n16,
         rtol=1e-6, atol=1e-6)
    layer_e16 = nn.TransformerEncoderLayer(D_16, H_16, F_16, dropout=0.0, activation="gelu", batch_first=True,
                                           norm_first=True, layer_norm_eps=0.1).double().eval()
    layer_e16.load_state_dict(layer_16.state_dict())
    with torch.no_grad():
        want_e16 = layer_e16(torch.from_numpy(x_16)).numpy()
    same("B3.16", "eps=0.1 (les deux layer norms)", A.transformer_block(x_16, params_16, H_16, eps=0.1), want_e16,
         rtol=1e-6, atol=1e-6)
    verdict("B3.16", np.array_equal(x_16, x_copy_16) and all(np.array_equal(params_16[k], saved_16[k]) for k in saved_16),
            "x et les paramètres reçus n'ont pas changé.", "transformer_block a modifié x ou les paramètres reçus.")
    no_w1_16 = {name: p for name, p in params_16.items() if name != "W_1"}
    refuses("B3.16", "transformer_block sans W_1 (ValueError, pas KeyError)", lambda: A.transformer_block(x_16, no_w1_16, H_16))

In [ ]:
if written(lambda: mylearn.attention.transformer_block(np.zeros((1, 1, 2)), {}, 1)):
    run_pytest_group(TEST_ATT, "test_transformer_block_")   # 25 tests
else:
    print("⏳ Ex B3.16 : les tests attendent ta fonction transformer_block.")

### Ex B3.17 — Trois masques qui mentent : convention inversée, masque transposé, padding compté dans la loss 🐛 ★★ ⏱️ 25 min
**Objectif :** trouver et corriger les trois erreurs de masque d'un petit modèle de langue, guidé par des diagnostics enchaînés.  
**Prérequis :** Ex B3.12 · fiche §203.3 (encadrés 🕰️ sur `F.scaled_dot_product_attention` et `nn.MultiheadAttention`), §203.8 (« La loss et la perplexité ») · tableau des pièges ⚠️ · **Fil rouge :** Holmes · **Parcours :** R, C

Une collègue a écrit un petit modèle de langue au niveau caractère : une couche d'attention causale sur les embeddings des caractères et de leurs positions, puis une couche dense. Elle calculait d'abord l'attention avec `F.scaled_dot_product_attention`, puis elle est passée à `nn.MultiheadAttention` pour lire les poids d'attention. Le modèle s'entraîne sur des lignes de Holmes de longueurs différentes, complétées par du padding. Son code contient **trois** erreurs. Corrige-le **sur place**, dans sa cellule, sans changer les noms, les signatures ni ce que rendent les fonctions.

Ce qu'il devrait faire :
- `causal_mask_17(T)` : le masque causal `(T, T)` dans la convention de `F.scaled_dot_product_attention`, `True` là où la requête $i$ peut regarder la clé $j$ ;
- `TinyLM_17` : des ids `(N, T)` → des logits `(N, T, V)`, chaque position ne voyant que les caractères jusqu'à elle ; `self.weights` garde les poids d'attention (la moyenne des deux têtes) ;
- `lm_loss_17(logits, targets)` : la cross-entropy moyenne sur les **vrais** caractères du batch ; les cibles valent `PAD_17` sur le padding.

La cellule donnée prépare les données : `LINES_17` (des lignes de l'entraînement de Holmes, de 24 à 64 caractères), `batch_17` (un batch de lignes, entrées et cibles décalées d'un caractère, complétées par `PAD_17`), `VAL_17` (64 fenêtres de 64 caractères de la validation, sans padding). Elle définit aussi `diagnose_17`. La cellule de vérification lance `diagnose_17`, trois diagnostics enchaînés, chacun ne s'exécutant que si le précédent passe, car une erreur peut en cacher une autre :
- la loss d'un batch est-elle un nombre ?
- le modèle voit-il le futur ? (un caractère changé, puis 200 itérations d'entraînement, quelques secondes)
- la loss d'un batch dépend-elle de son padding ?

Elle tourne dès le premier « Tout exécuter » : ses messages sont ton point de départ. Une fois les trois points passés, la cellule suivante trace les poids d'attention de ta version réparée sur une ligne de Holmes.

Dans tes notes : les trois erreurs, et pour chacune le symptôme qu'elle produirait seule. Pourquoi les `nan` n'apparaissent-ils qu'à une seule position, et laquelle, selon que l'autre erreur de masque est encore là ou non ? Laquelle des trois erreurs fait baisser la loss d'entraînement sans que le modèle prédise mieux le texte ?

In [ ]:
PAD_17 = len(DATA.itos)                    # one more id for the padding: 89 ids in all
VOCAB_17, LEN_17 = PAD_17 + 1, 64
LINES_17 = [line for line in DATA.train_text.split("\n") if 24 <= len(line) <= LEN_17]    # lines of Holmes
VAL_17 = DATA.val_ids[:LEN_17 * 64].view(64, LEN_17)                                       # 64 windows, no padding


def batch_17(lines, generator, batch_size=32):
    """A batch of lines of Holmes: inputs ids[:-1] and targets ids[1:], padded with PAD_17 to LEN_17 - 1 positions."""
    picks = torch.randint(len(lines), (batch_size,), generator=generator).tolist()
    width = max(len(lines[i]) for i in picks) - 1
    x = torch.full((batch_size, width), PAD_17)
    y = torch.full((batch_size, width), PAD_17)
    for row, i in enumerate(picks):
        ids = DATA.encode(lines[i])
        x[row, :len(ids) - 1], y[row, :len(ids) - 1] = ids[:-1], ids[1:]
    return x, y


def diagnose_17(model_class, loss_fn, iters=200):
    """Three diagnoses of the small language model, in a row: the first one that fails stops the chain (one error can
    hide another). Returns the trained model when the three pass, else None."""
    torch.manual_seed(17)
    model = model_class(VOCAB_17)
    gen = torch.Generator().manual_seed(17)
    x, y = batch_17(LINES_17, gen)

    def point1():
        with torch.no_grad():
            logits = model(x)
            loss = loss_fn(logits, y)
        if torch.isfinite(loss) and torch.isfinite(logits).all():
            return True, f"la loss d'un batch à l'initialisation est un nombre, {fr(loss.item(), 3)}", ""
        bad = sorted({int(t) for t in torch.nonzero(~torch.isfinite(logits).all(dim=-1))[:, 1]})
        return False, (f"la loss d'un batch vaut {loss.item()} ; les logits sont nan à la position {bad[0]} de chaque séquence"
                       if len(bad) == 1 else f"la loss d'un batch vaut {loss.item()} ; logits nan aux positions {bad[:6]}"), \
            "que voit cette position dans la couche d'attention ?"

    def point2():
        probe = VAL_17[:4].clone()
        changed = probe.clone()
        changed[:, 40] = (changed[:, 40] + 1) % PAD_17                      # change the character 40 of each window
        with torch.no_grad():
            gap = (model(probe)[:, :40] - model(changed)[:, :40]).abs().max().item()
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
        for _ in range(iters):
            xb, yb = batch_17(LINES_17, gen)
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        with torch.no_grad():
            val = F.cross_entropy(model(VAL_17[:, :-1]).flatten(0, 1), VAL_17[:, 1:].flatten()).item()
        if gap < 1e-6:
            return True, (f"changer le caractère 40 d'une fenêtre ne change aucun logit des positions 0 à 39 ; après "
                          f"{iters} itérations, la loss de validation vaut {fr(val, 2)} nats par caractère"), ""
        return False, (f"changer le caractère 40 d'une fenêtre change les logits des positions 0 à 39 (écart maximal "
                       f"{gap:.2g}) ; après {iters} itérations, la loss de validation tombe à {fr(val, 2)} nat par "
                       "caractère"), "pour un modèle d'une seule couche, est-ce crédible ? D'où vient l'information ?"

    def point3():
        short = [line for line in LINES_17 if len(line) <= 40][:16]
        width = max(len(line) for line in short) - 1
        x_s = torch.full((16, width), PAD_17)
        y_s = torch.full((16, width), PAD_17)
        for row, line in enumerate(short):
            ids = DATA.encode(line)
            x_s[row, :len(ids) - 1], y_s[row, :len(ids) - 1] = ids[:-1], ids[1:]
        extra = 60 - width                                                   # the same lines, 60 positions wide
        x_l = torch.cat([x_s, torch.full((16, extra), PAD_17)], dim=1)
        y_l = torch.cat([y_s, torch.full((16, extra), PAD_17)], dim=1)
        with torch.no_grad():
            short_loss, long_loss = loss_fn(model(x_s), y_s).item(), loss_fn(model(x_l), y_l).item()
        if abs(short_loss - long_loss) < 1e-5:
            return True, (f"la loss d'un même batch ne dépend pas de son padding ({fr(short_loss, 4)} sur {width} "
                          f"positions comme sur 60)"), ""
        return False, (f"la loss d'un même batch de 16 lignes vaut {fr(short_loss, 4)} sur {width} positions et "
                       f"{fr(long_loss, 4)} une fois complété par {extra} positions de padding"), \
            "quelles positions entrent dans la moyenne ?"

    points = [("la loss est un nombre", point1), ("le modèle ne voit pas le futur", point2),
              ("le padding ne compte pas", point3)]
    for k, (name, run) in enumerate(points, start=1):
        ok, measured, why = run()
        if ok:
            print(f"✅ Ex B3.17 : point {k}, {name} ({measured}).")
        else:
            print(f"❌ Ex B3.17 : point {k} : {measured} ; {why}")
            for later in range(k + 1, len(points) + 1):
                print(f"⏳ Ex B3.17 : point {later} : corrige d'abord le point {k}.")
            return None
    print("✅ Ex B3.17 : les trois diagnostics passent.")
    return model

In [ ]:
# The small language model to repair IN PLACE (same names, same signatures, same returns): the diagnostics of the
# next cell start from it. Its author first computed the attention with F.scaled_dot_product_attention, then switched
# to nn.MultiheadAttention to read the attention weights.
def causal_mask_17(T):
    """Boolean causal mask in the convention of F.scaled_dot_product_attention: True where query i may attend to key j."""
    return torch.ones(T, T, dtype=torch.bool).triu()


class TinyLM_17(nn.Module):
    """One causal attention layer over character and position embeddings, then a dense layer: ids (N, T) -> logits
    (N, T, vocab_size)."""

    def __init__(self, vocab_size, d_model=32, n_head=2, max_len=64):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_head, batch_first=True)
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, ids):
        T = ids.shape[1]
        x = self.tok(ids) + self.pos(torch.arange(T))
        h, self.weights = self.attn(x, x, x, attn_mask=causal_mask_17(T))     # weights kept to draw the maps
        return self.head(self.ln(x + h))


def lm_loss_17(logits, targets):
    """Mean cross-entropy over the real characters of a batch: logits (N, T, V), targets (N, T) with PAD_17 on the
    padding."""
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))

In [ ]:
with checked("B3.17"):
    model_17 = diagnose_17(TinyLM_17, lm_loss_17)

In [ ]:
if globals().get("model_17") is None:
    print("⏳ Ex B3.17 : la carte attend que les trois points de la cellule précédente passent.")
else:
    with checked("B3.17"):
        line_17 = "I had seen little of Holmes lately."                  # a line of the first story
        ids_17 = DATA.encode(line_17)[None, :-1]
        model_17.eval()
        with torch.no_grad():
            logits_17 = model_17(ids_17)
        w_17 = model_17.weights[0].numpy()                                 # (T, T), averaged over the 2 heads
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.imshow(w_17, cmap="viridis")
        ax.set_xticks(range(len(line_17) - 1), list(line_17[:-1]))
        ax.set_yticks(range(len(line_17) - 1), list(line_17[:-1]))
        ax.set_xlabel("clé")
        ax.set_ylabel("requête")
        ax.set_title("ta version réparée : poids d'attention (moyenne des deux têtes)")
        plt.show()
        print("au-dessus de la diagonale, le plus grand poids vaut", fr(float(np.triu(w_17, 1).max()), 6))

## Partie B · Le mini-GPT en PyTorch

Tu passes de NumPy à PyTorch : le module d'attention causale du mini-GPT, comparé à ta version NumPy (B3.18), le modèle complet (B3.19), puis son entraînement sur Holmes, face au LSTM du ch. 22 (B3.20). Le modèle suit la structure de nanoGPT (A. Karpathy, licence MIT, fiche §203.8) ; le contrat interdit `nn.Transformer…` et `nn.MultiheadAttention` dans ton modèle.

Le modèle que B3.20 entraîne est **sauvegardé une fois** (son `state_dict`, sa configuration, le vocabulaire et l'historique de l'entraînement), hors du dépôt, sous une clé qui dit tout ce qui l'a produit (configuration, réglages, graine, mode rapide ou complet, empreinte du découpage) : la suite du chapitre le recharge sans réentraîner. Ton modèle doit donc suivre les noms du contrat de B3.19.

La cellule suivante charge les outils de la partie : `MINI_CONFIG` (4 blocs, 4 têtes, $d = 128$, contexte de 128 caractères, 88 caractères), `n_params`, `forbidden_in` (les modules interdits trouvés dans un modèle), `params_from_qkv` (les poids de ton module d'attention dans la convention de `multi_head_attention`). Le vocabulaire est celui de ton `build_vocab` (ch. 22) : les caractères par fréquence décroissante, à égalité dans l'ordre de `sorted` ; la cellule vérifie que ta fonction donne le même. `wb.gpt` contient aussi une solution de référence de B3.18 à B3.20 (`ReferenceGPT`, `train`) : le notebook ne s'en sert que pour vérifier tes fonctions ou quand ton modèle manque ; ne la lis qu'une fois ta version écrite.

In [ ]:
# Tools of part B: the mini-GPT in PyTorch
MINI_CONFIG = dict(wbgpt.MINI_CONFIG)          # 4 blocks, 4 heads, d = 128, context of 128 characters, 88 ids
FORBIDDEN = (nn.MultiheadAttention, nn.Transformer, nn.TransformerEncoder, nn.TransformerDecoder,
             nn.TransformerEncoderLayer, nn.TransformerDecoderLayer)


def n_params(model):
    """The number of parameters of a module, each tied parameter counted once (model.parameters() lists it once)."""
    return sum(p.numel() for p in model.parameters())


def forbidden_in(model):
    """The names of the ready-made attention modules of PyTorch found in a model (the contract forbids them)."""
    return sorted({type(m).__name__ for m in model.modules() if isinstance(m, FORBIDDEN)})


def params_from_qkv(qkv, proj):
    """The parameters of an attention module (qkv: nn.Linear(d, 3d), queries then keys then values; proj:
    nn.Linear(d, d)) in the convention of multi_head_attention (z = x @ W + b, float64): W = weight.T."""
    D = proj.in_features
    W = qkv.weight.detach().double().numpy()
    b = qkv.bias.detach().double().numpy()
    return {"W_q": W[:D].T.copy(), "W_k": W[D:2 * D].T.copy(), "W_v": W[2 * D:].T.copy(),
            "W_o": proj.weight.detach().double().numpy().T.copy(), "b_q": b[:D].copy(), "b_k": b[D:2 * D].copy(),
            "b_v": b[2 * D:].copy(), "b_o": proj.bias.detach().double().numpy().copy()}


try:                                                   # the vocabulary: the rule of your build_vocab (ch. 22, 22.12)
    _, itos_vocab = mylearn.sequences.build_vocab(DATA.train_text + DATA.val_text)
    print("ℹ️ ton build_vocab (ch. 22) donne le vocabulaire de DATA :" if list(itos_vocab) == DATA.itos else
          "⚠️ ton build_vocab (ch. 22) ne donne pas le vocabulaire de DATA (décroissance de la fréquence, ex aequo dans "
          "l'ordre de sorted) : le notebook garde celui de DATA, relance les tests du ch. 22 ;",
          f"{len(DATA.itos)} caractères, les plus fréquents {DATA.itos[:6]}.")
except Exception:                                     # not written yet (NotImplementedError), or no module at all
    print(f"ℹ️ ton build_vocab (ch. 22, 22.12) n'est pas écrit : le notebook prend le vocabulaire de DATA, construit par "
          f"la même règle ({len(DATA.itos)} caractères).")

### Ex B3.18 — De NumPy à PyTorch : un module CausalSelfAttention avec F.scaled_dot_product_attention 📦 ★★ ⏱️ 30 min
**Objectif :** écrire le module d'attention causale du mini-GPT en PyTorch, et mesurer son écart à ta version NumPy.  
**Prérequis :** Ex B3.14 · ch. 20 (`nn.Module`, dropout) · fiche §203.3 (encadré 🕰️ sur `F.scaled_dot_product_attention`), §203.4 · **Parcours :** R, M, C

**Contrat** de `CausalSelfAttention_18(d_model, n_head, dropout=0.0)`, un `nn.Module` :
- exactement deux couches denses, avec biais : `qkv = nn.Linear(d_model, 3 * d_model)`, dont les sorties sont, dans l'ordre, les requêtes, les clés puis les valeurs ; et `proj = nn.Linear(d_model, d_model)`, la projection de sortie $\mathbf{W}_O$ ; `ValueError` si `n_head` ne divise pas `d_model` ;
- `forward(x)` : `x` de forme `(N, T, d_model)` → `(N, T, d_model)` : le découpage en `n_head` têtes comme ton `split_heads` (B3.14), l'attention causale par `F.scaled_dot_product_attention` avec son échelle par défaut, le recollage des têtes comme `merge_heads`, puis `proj` ;
- dropout : `dropout` sur les poids d'attention (`dropout_p`) et un `nn.Dropout(dropout)` sur la sortie de `proj`, en mode train seulement ; en mode eval, aucun dropout (`dropout_p = 0`, fiche §203.3) ;
- ni `nn.MultiheadAttention` ni `nn.Transformer…`.

La cellule de vérification contrôle les noms et les formes des paramètres, puis compare ta sortie, en mode eval, à celle de `nn.MultiheadAttention` aux mêmes poids, avec le masque causal (l'oracle de PyTorch), et à ta `multi_head_attention` NumPy de B3.14 avec les mêmes poids, convertis par `params_from_qkv` (`W = weight.T`). L'écart maximal entre ton module (float32) et ta version NumPy (float64) est vérifié : il doit rester sous $10^{-5}$ (rien à taper). Elle vérifie enfin la causalité, puis le dropout en mode train et en mode eval.

Dans tes notes : quel écart maximal mesures-tu entre tes deux versions, et d'où vient-il ? Pourquoi `view` juste après `transpose` lève-t-il une erreur, et que fait `reshape` à la place (fiche §203.4) ? Que se passerait-il en passant à la fois un masque et `is_causal=True` à SDPA (fiche §203.3) ?

In [ ]:
class CausalSelfAttention_18(nn.Module):
    """Causal multi-head self-attention (contract of Ex B3.18): x (N, T, d_model) -> (N, T, d_model)."""

    def __init__(self, d_model, n_head, dropout=0.0):
        super().__init__()
        raise NotImplementedError("CausalSelfAttention_18")

    def forward(self, x):
        raise NotImplementedError("CausalSelfAttention_18.forward")

In [ ]:
attn_ok_18 = False
with checked("B3.18"):
    torch.manual_seed(18)
    attn_18 = CausalSelfAttention_18(32, 4)
    shapes_18 = {name: tuple(p.shape) for name, p in attn_18.named_parameters()}
    want_18 = {"qkv.weight": (96, 32), "qkv.bias": (96,), "proj.weight": (32, 32), "proj.bias": (32,)}
    verdict("B3.18", shapes_18 == want_18, "deux couches denses, qkv (32 → 96) et proj (32 → 32), avec leurs biais.",
            f"paramètres attendus {want_18}, reçus {shapes_18} : relis le contrat (noms et formes).")
    if shapes_18 != want_18:
        raise StopCheck
    if forbidden_in(attn_18):
        print(f"❌ Ex B3.18 : ton module contient {forbidden_in(attn_18)} : le contrat demande F.scaled_dot_product_attention.")
        raise StopCheck
    attn_18.eval()
    x_18 = torch.randn(3, 10, 32)
    with torch.no_grad():
        y_18 = attn_18(x_18)
    if tuple(y_18.shape) != (3, 10, 32):
        print(f"❌ Ex B3.18 : x (3, 10, 32) donne une sortie {tuple(y_18.shape)} ; attendu (3, 10, 32).")
        raise StopCheck
    # the same attention in PyTorch's ready-made module, with the same weights: the oracle that is not yours
    params_18 = params_from_qkv(attn_18.qkv, attn_18.proj)
    mha_18 = nn.MultiheadAttention(32, 4, batch_first=True)
    with torch.no_grad():
        mha_18.in_proj_weight.copy_(attn_18.qkv.weight)
        mha_18.in_proj_bias.copy_(attn_18.qkv.bias)
        mha_18.out_proj.weight.copy_(attn_18.proj.weight)
        mha_18.out_proj.bias.copy_(attn_18.proj.bias)
        torch_18, _ = mha_18(x_18, x_18, x_18, attn_mask=torch.from_numpy(~causal_np(10)), need_weights=False)
    gap_torch_18 = (y_18 - torch_18).abs().max().item()
    verdict("B3.18", gap_torch_18 < 1e-5,
            f"ton module donne la sortie de nn.MultiheadAttention aux mêmes poids (écart maximal {gap_torch_18:.1e}).",
            f"ton module s'écarte de {gap_torch_18:.2g} de nn.MultiheadAttention aux mêmes poids (masque causal) : vérifie "
            "l'ordre q, k, v dans qkv, le découpage en têtes (view puis transpose), le masque causal et le recollage des têtes.")
    # your NumPy version (Ex B3.14) with the same weights, in float64
    x64_18 = x_18.double().numpy()
    try:
        out_np_18 = np.asarray(mylearn.attention.multi_head_attention(x64_18, x64_18, x64_18, params_18, 4,
                                                                      mask=causal_np(10))[0], dtype=float)
    except (NotImplementedError, ImportError, AttributeError):      # not written yet, or no attention.py at all
        out_np_18 = None
        print("⏳ Ex B3.18 : l'écart à ta version NumPy attend ta multi_head_attention (B3.14).")
    except Exception as exc:                                         # noqa: BLE001 - a bug of B3.14: say it, go on
        out_np_18 = None
        print(f"❌ Ex B3.18 : ta multi_head_attention (B3.14) lève {type(exc).__name__} : {str(exc)[:160]} ; relance ses tests.")
    if out_np_18 is not None and out_np_18.shape != (3, 10, 32):
        print(f"❌ Ex B3.18 : ta multi_head_attention (B3.14) rend une sortie {out_np_18.shape} au lieu de (3, 10, 32) ; "
              "relance ses tests.")
        out_np_18 = None
    if out_np_18 is not None:
        gap_18 = float(np.abs(y_18.double().numpy() - out_np_18).max())
        print(f"écart maximal entre ton module PyTorch (float32) et ta multi_head_attention NumPy (float64) : {gap_18:.1e}")
        if gap_18 >= 1e-5 and gap_torch_18 < 1e-5:
            print("❌ Ex B3.18 : ton module est d'accord avec nn.MultiheadAttention : c'est ta multi_head_attention (B3.14) "
                  "qui s'en écarte ; relance ses tests.")
        wb.check("B3.18", bool(gap_18 < 1e-5), computed=True)
    # causality, then dropout only in training
    x2_18 = x_18.clone()
    x2_18[:, 6] += 1.0
    with torch.no_grad():
        past_18 = (attn_18(x2_18)[:, :6] - y_18[:, :6]).abs().max().item()
    verdict("B3.18", past_18 == 0, "changer le token 6 ne change aucune sortie des tokens 0 à 5 : le module est causal.",
            f"changer le token 6 change les sorties des tokens 0 à 5 (écart {past_18:.2g}) : le module voit le futur.")
    drop_18 = CausalSelfAttention_18(32, 4, dropout=0.5)
    drop_18.load_state_dict(attn_18.state_dict())
    with torch.no_grad():
        drop_18.train()
        a_18, b_18 = drop_18(x_18), drop_18(x_18)
        drop_18.eval()
        c_18 = drop_18(x_18)
    verdict("B3.18", not torch.allclose(a_18, b_18) and torch.allclose(c_18, y_18),
            "avec dropout=0.5 : deux passes différentes en mode train, la sortie sans dropout en mode eval.",
            "avec dropout=0.5, attendu deux sorties différentes en mode train (dropout actif) et, en mode eval, la sortie "
            "du module sans dropout (dropout_p = 0 hors entraînement).")
    attn_ok_18 = gap_torch_18 < 1e-5 and past_18 == 0

### Ex B3.19 — Mini-GPT caractère par caractère : embeddings de tokens et de positions, blocs pré-LN, tête liée 📦 ★★★ ⏱️ 60 min
**Objectif :** assembler le mini-GPT complet en PyTorch, avec la structure et l'initialisation de nanoGPT.  
**Prérequis :** Ex B3.18 · Ex B3.15 · ✏️ B3.5 j) · fiche §203.6, §203.7 (« Le décodeur : GPT »), §203.8 (« Le modèle », « La recette d'entraînement ») · **Fil rouge :** Holmes · **Parcours :** R, M, C

**Contrat.** Les noms des modules comptent : B3.20 sauvegarde ton modèle par son `state_dict`, et la suite du chapitre le recharge et lit ses blocs.
- `Block_19(d_model, n_head, dropout=0.0)`, un bloc pré-LN : `ln1` et `ln2` (`nn.LayerNorm(d_model)`), `attn` (ton `CausalSelfAttention_18`), `mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model), nn.Dropout(dropout))` ; `forward(x)` : d'abord `x + attn(ln1(x))`, puis on ajoute `mlp(ln2(·))` du résultat.
- `MiniGPT_19(vocab_size, block_size=128, n_layer=4, n_head=4, d_model=128, dropout=0.0)` :
  - `tok_emb` (`nn.Embedding(vocab_size, d_model)`), `pos_emb` (des positions apprises, `nn.Embedding(block_size, d_model)`), un `nn.Dropout(dropout)` sur leur somme, `blocks` (un `nn.ModuleList` de `n_layer` `Block_19`), `ln_f` (`nn.LayerNorm(d_model)`) et `head` (`nn.Linear(d_model, vocab_size, bias=False)`) ;
  - **poids liés** : `head.weight` **est** `tok_emb.weight`, le même tenseur, compté une fois ;
  - **initialisation de nanoGPT** (fiche §203.8) : les poids des couches denses et des deux tables selon $\mathcal{N}(0;\ 0{,}02^2)$, les biais à zéro, et les deux projections qui écrivent dans le chemin résiduel de chaque bloc, `attn.proj` et `mlp[2]`, selon $\mathcal{N}\big(0;\ (0{,}02 / \sqrt{2\,n_{\text{layer}}})^2\big)$ ; les layer norms gardent $\gamma = 1$ et $\beta = 0$ ;
  - `self.block_size` garde `block_size` ; `forward(idx)` : des ids `(N, T)` int64 → des logits `(N, T, vocab_size)` ; `ValueError` si `T > block_size` ;
  - le `state_dict` a les noms et les formes de `wbgpt.expected_state_shapes()` (affiche-le pour comparer).
- Ni `nn.Transformer…` ni `nn.MultiheadAttention`.

La cellule de vérification construit ton modèle avec la configuration du contrat (`MINI_CONFIG`) et contrôle : les noms et les formes de son `state_dict`, les poids liés, son nombre de paramètres (celui que tu as calculé en ✏️ B3.5 j ; la cellule ne le redonne pas), la présence de tes `Block_19` et `CausalSelfAttention_18`, le bloc pré-LN, l'initialisation, la forme des logits, l'usage des positions, la loss à l'initialisation (le contrôle de départ de la fiche), la causalité et le refus d'une séquence trop longue.

Dans tes notes : mesure la loss de départ d'un modèle dont les tables gardent l'initialisation par défaut de PyTorch, $\mathcal{N}(0;\ 1)$ pour `nn.Embedding` : que se passe-t-il, et pourquoi la tête liée l'aggrave-t-elle ? Pourquoi réduire l'initialisation des deux projections résiduelles quand le nombre de blocs grandit ?

In [ ]:
class Block_19(nn.Module):
    """Pre-LN Transformer block of the mini-GPT (contract of Ex B3.19): x (N, T, d_model) -> (N, T, d_model)."""

    def __init__(self, d_model, n_head, dropout=0.0):
        super().__init__()
        raise NotImplementedError("Block_19")

    def forward(self, x):
        raise NotImplementedError("Block_19.forward")


class MiniGPT_19(nn.Module):
    """Character-level mini-GPT (contract of Ex B3.19): idx (N, T) int64 with T <= block_size -> logits
    (N, T, vocab_size)."""

    def __init__(self, vocab_size, block_size=128, n_layer=4, n_head=4, d_model=128, dropout=0.0):
        super().__init__()
        raise NotImplementedError("MiniGPT_19")

    def forward(self, idx):
        raise NotImplementedError("MiniGPT_19.forward")

In [ ]:
model_ok_19 = False
with checked("B3.19"):
    torch.manual_seed(19)
    gpt_19 = MiniGPT_19(**MINI_CONFIG)
    if forbidden_in(gpt_19):
        print(f"❌ Ex B3.19 : ton modèle contient {forbidden_in(gpt_19)} : le contrat interdit les modules d'attention tout faits.")
        raise StopCheck
    got_19 = {name: tuple(t.shape) for name, t in gpt_19.state_dict().items()}
    want_19 = wbgpt.expected_state_shapes(MINI_CONFIG)
    missing_19 = [n for n in want_19 if n not in got_19]
    extra_19 = [n for n in got_19 if n not in want_19]
    wrong_19 = [f"{n} {got_19[n]} au lieu de {want_19[n]}" for n in want_19 if n in got_19 and got_19[n] != want_19[n]]
    verdict("B3.19", not (missing_19 or extra_19 or wrong_19),
            f"les {len(want_19)} tenseurs du state_dict ont les noms et les formes du contrat.",
            "le state_dict ne suit pas le contrat : " + "; ".join(filter(None, [
                f"manquants {missing_19[:4]}" if missing_19 else "", f"en trop {extra_19[:4]}" if extra_19 else "",
                f"formes {wrong_19[:3]}" if wrong_19 else ""])) + ".")
    if missing_19 or extra_19 or wrong_19:
        raise StopCheck
    tied_19 = gpt_19.head.weight is gpt_19.tok_emb.weight
    verdict("B3.19", tied_19, "la couche de sortie et la table des tokens sont le même tenseur (poids liés).",
            "head.weight doit être le tenseur tok_emb.weight lui-même (poids liés), pas une copie.")
    verdict("B3.19", n_params(gpt_19) == wbgpt.count_params(MINI_CONFIG),
            "le même nombre de paramètres que ton calcul de ✏️ B3.5 j) (la tête liée comptée une fois).",
            f"ton modèle a {n_params(gpt_19):_} paramètres : ce n'est pas le compte de ✏️ B3.5 j) ; relis le contrat (biais, "
            "positions apprises, layer norm finale, tête liée sans biais).".replace("_", " "))
    kinds_19 = {type(b).__name__ for b in gpt_19.blocks} | {type(b.attn).__name__ for b in gpt_19.blocks}
    verdict("B3.19", kinds_19 == {"Block_19", "CausalSelfAttention_18"},
            "4 blocs Block_19, chacun avec ton CausalSelfAttention_18.",
            f"blocks doit contenir des Block_19, dont attn est un CausalSelfAttention_18 (B3.18) ; reçu {sorted(kinds_19)}.")
    blk_19 = gpt_19.blocks[0].eval()
    xb_19 = torch.randn(2, 5, MINI_CONFIG["d_model"])
    with torch.no_grad():
        h_19 = xb_19 + blk_19.attn(blk_19.ln1(xb_19))
        pre_ln_19 = torch.allclose(blk_19(xb_19), h_19 + blk_19.mlp(blk_19.ln2(h_19)), atol=1e-6)
    verdict("B3.19", pre_ln_19, "Block_19 est pré-LN : h = x + attn(ln1(x)), puis h + mlp(ln2(h)).",
            "Block_19(x) ne vaut pas h + mlp(ln2(h)) avec h = x + attn(ln1(x)) : relis le bloc pré-LN du contrat (§203.6).")
    std_19 = {"tok_emb": gpt_19.tok_emb.weight.std().item(), "attn.qkv": gpt_19.blocks[0].attn.qkv.weight.std().item(),
              "attn.proj": gpt_19.blocks[0].attn.proj.weight.std().item(), "mlp.2": gpt_19.blocks[0].mlp[2].weight.std().item()}
    want_std_19 = {"tok_emb": 0.02, "attn.qkv": 0.02, "attn.proj": 0.02 / math.sqrt(8), "mlp.2": 0.02 / math.sqrt(8)}
    zero_bias_19 = all(m.bias.detach().abs().max().item() == 0 for m in gpt_19.modules() if isinstance(m, nn.Linear) and m.bias is not None)
    verdict("B3.19", all(abs(std_19[k] / want_std_19[k] - 1) < 0.1 for k in std_19) and zero_bias_19,
            "initialisation de nanoGPT : écarts-types 0,02 et 0,02/√8 pour les deux projections résiduelles, biais nuls.",
            "l'initialisation n'est pas celle du contrat : écarts-types mesurés "
            + ", ".join(f"{k} {fr(v, 4)}" for k, v in std_19.items()) + (", biais non nuls" if not zero_bias_19 else "") + ".")
    x_19, y_19 = wbgpt.get_batch(DATA.train_ids, 4, 128, torch.Generator().manual_seed(19))
    gpt_19.eval()
    with torch.no_grad():
        logits_19 = gpt_19(x_19)
        shape_ok_19 = tuple(logits_19.shape) == (4, 128, 88)
        verdict("B3.19", shape_ok_19, "des ids (4, 128) donnent des logits (4, 128, 88).",
                f"des ids (4, 128) donnent {tuple(logits_19.shape)} : attendu (4, 128, 88), un vecteur de logits par position.")
        if not shape_ok_19:
            raise StopCheck
        loss0_19 = F.cross_entropy(logits_19.flatten(0, 1), y_19.flatten()).item()
        verdict("B3.19", abs(loss0_19 - math.log(88)) < 0.1,
                "à l'initialisation, la loss est celle d'une prédiction presque uniforme (le contrôle de départ, §203.8).",
                f"à l'initialisation, la loss d'un batch vaut {fr(loss0_19, 2)} : loin d'une prédiction presque uniforme ; "
                "relis le contrôle de départ (§203.8) et l'initialisation du contrat.")
        repeated_19 = gpt_19(torch.zeros(1, 16, dtype=torch.long))[0]
        positions_19 = (repeated_19[1:] - repeated_19[:1]).abs().max().item() > 1e-6
        verdict("B3.19", positions_19,
                "les positions apprises servent : un même caractère répété 16 fois donne des logits qui changent d'une "
                "position à l'autre.",
                "un même caractère répété 16 fois donne les mêmes logits à chaque position : les positions apprises "
                "sont-elles ajoutées aux embeddings des tokens ?")
        changed_19 = x_19.clone()
        changed_19[:, 100] = (changed_19[:, 100] + 1) % 88
        future_19 = (gpt_19(changed_19)[:, :100] - logits_19[:, :100]).abs().max().item()
    verdict("B3.19", future_19 == 0, "changer le caractère 100 ne change aucun logit des positions 0 à 99 : le modèle est causal.",
            f"changer le caractère 100 change les logits des positions 0 à 99 (écart {future_19:.2g}) : le modèle voit le futur.")
    refuses("B3.19", "129 caractères pour un contexte de 128", lambda: gpt_19(torch.zeros(1, 129, dtype=torch.long)))
    model_ok_19 = all([tied_19, n_params(gpt_19) == wbgpt.count_params(MINI_CONFIG), pre_ln_19, positions_19,
                       kinds_19 == {"Block_19", "CausalSelfAttention_18"}, abs(loss0_19 - math.log(88)) < 0.1,
                       future_19 == 0])

### Ex B3.20 — Entraîner le mini-GPT sur Holmes : AdamW, warmup + cosinus, clipping, perplexité, face au LSTM du ch. 22 📦 ★★★ ⏱️ 60 min 🚀
**Objectif :** entraîner le mini-GPT avec la recette de nanoGPT, mesurer sa perplexité de validation et la comparer à celle du LSTM du ch. 22.  
**Prérequis :** Ex B3.19 · ch. 19 (AdamW, warmup + cosinus, clipping ; 19.19) · ch. 22 (22.25, 22.26) · fiche §203.8 (« Les données », « La loss et la perplexité », « La recette d'entraînement ») · **Fil rouge :** Holmes · **Parcours :** R, M, C

Les données : `DATA.train_ids` et `DATA.val_ids`, deux tenseurs 1-D d'ids (le découpage figé de `wb.text.train_val("holmes")` : 90 % puis 10 %, coupé à une fin de ligne, jamais mélangé : la validation n'est jamais vue) ; `get_batch`, celui de la fiche ; `CONFIG_20`, la configuration du modèle (celle de B3.19, avec un dropout de 0,2 en mode complet : 5 000 itérations font environ 40 passages sur le texte d'entraînement, aucun en mode rapide) ; `TRAIN_20`, les réglages (affichés par la cellule suivante).

**Contrat**
- `configure_optimizer_20(model, lr, weight_decay, betas=(0.9, 0.99))` : un `torch.optim.AdamW` à deux groupes, `weight_decay` sur les paramètres à deux axes ou plus (les matrices et les tables d'embedding), 0 sur les autres (les biais, les $\gamma$ et $\beta$ des layer norms) ; chaque paramètre une seule fois.
- `estimate_loss_20(model, data, n_batches, batch_size, block_size, generator)` : la moyenne, en float Python, des loss de `n_batches` batches `get_batch(data, batch_size, block_size, generator)`, en mode eval et sans gradient ; le modèle revient ensuite en mode train.
- `train_20(model, train_ids, val_ids, settings, generator, block_size=128)` entraîne `model` sur place :
  - avec l'optimiseur de `configure_optimizer_20(model, settings["lr"], settings["weight_decay"], (0.9, settings["beta2"]))` ;
  - à chaque itération `it`, de 0 à `iters − 1`, dans cet ordre : le learning rate de tous les groupes, `cosine_lr(it, iters, lr, warmup, min_lr)` de ton ch. 19 (19.19 : montée linéaire depuis 0 pendant `warmup` itérations, puis cosinus jusqu'à `min_lr` ; `mylearn.optim.cosine_lr`, ou la même formule) ; un batch `get_batch(train_ids, batch_size, block_size, generator)` ; la loss, la cross-entropy moyenne sur les $N \times T$ positions (fiche §203.8) ; les gradients ; le clipping à la norme `settings["clip"]` ; le pas de l'optimiseur ;
  - l'évaluation, `estimate_loss_20` sur `val_ids` avec `eval_batches` batches, avant la première itération, après chaque multiple de `eval_interval` itérations et après la dernière ; chacune tire ses batches avec un générateur neuf, `torch.Generator().manual_seed(settings["eval_seed"])` (les mêmes batches de validation à chaque fois ; le générateur de l'entraînement ne sert jamais à l'évaluation) ;
  - rend l'historique, un dictionnaire de listes : `"train_loss"` et `"lr"` (une valeur par itération), `"val_iter"` (le nombre d'itérations faites à chaque évaluation : 0, 50, 100…) et `"val_loss"`.

La cellule de vérification contrôle d'abord tes trois fonctions sur un petit modèle (celui de référence de `wb.gpt`, pour ne pas dépendre de B3.19) : les groupes de l'optimiseur, paramètre par paramètre ; `estimate_loss_20` (le mode, le gradient, la valeur) ; puis 6 itérations de `train_20`, comparées à la recette du contrat, avec un clipping qui agit à chaque itération. La cellule suivante entraîne ton `MiniGPT_19` sur Holmes : 2 à 3 min sur un CPU à deux cœurs en mode rapide (600 itérations) ; 🚀 en mode complet (5 000 itérations avec dropout, environ 35 min sur CPU), un GPU de Colab est conseillé. La dernière mesure ensuite sa loss sur **toute** la validation (`wbgpt.char_loss` : chaque caractère à partir du 129ᵉ, prédit à partir des 97 à 128 caractères qui le précèdent), la compare au seuil `TARGET_20` et au LSTM du ch. 22 sur les mêmes caractères de validation (le tableau `LSTM_20`, précalculé), puis **sauvegarde** ton modèle dans `wbgpt.checkpoints_dir()` (sur ta machine, `data/downloads/gpt/`, hors du suivi de git ; sur Colab, `/content/wb_cache/gpt/`, qui s'efface avec la session). Le temps affiché est une mesure, pas une réponse : il dépend de la machine.

Dans tes notes : le mini-GPT bat-il le LSTM du ch. 22 à temps de calcul voisin ? Qu'est-ce qui rend la comparaison inégale (paramètres, contexte, caractères vus) ? Pourquoi la dernière loss estimée pendant l'entraînement (8 batches de validation) et celle de `char_loss` (toute la validation) diffèrent-elles un peu ?

In [ ]:
CONFIG_20 = wbgpt.mini_config(FAST_MODE)             # the model of B3.19; dropout 0.2 in full mode
TRAIN_20 = wbgpt.train_settings(FAST_MODE)           # 600 iterations in fast mode, 5 000 in full mode
SEED_20 = wbgpt.SEED                                 # seed of the initialisation and of the training batches
get_batch = wbgpt.get_batch                          # the get_batch of the fiche (§203.8)
print(CONFIG_20)
print(TRAIN_20)
TARGET_20 = 1.7 if FAST_MODE else 1.45   # nats per character on the whole validation (char_loss)
# The comparison (precomputed by tools/chapters/history_chB3.py on the same split, the same vocabulary and the same
# validation targets, every character from index 128 on): (model, parameters, budget, nats per character)
LSTM_20 = [
    ('LSTM du ch. 22, mode rapide', 255064, '2 epochs sur 240 000 car., 1,4 min', 1.7499665121943868),
    ('LSTM du ch. 22, mode complet (meilleure epoch)', 255064, "7 epochs sur tout l'entraînement, 9 min", 1.5431814462445237),
    ('mini-GPT de référence, mode complet (dropout 0,2)', 820992, '5 000 itérations, 34 min (CPU)', 1.3269839673867643),
]

In [ ]:
def configure_optimizer_20(model, lr, weight_decay, betas=(0.9, 0.99)):
    """torch.optim.AdamW with two parameter groups: weight_decay on the parameters with 2 axes or more, 0.0 on the
    others; each parameter exactly once."""
    raise NotImplementedError("configure_optimizer_20")


def estimate_loss_20(model, data, n_batches, batch_size, block_size, generator):
    """Mean cross-entropy (nats per token, a Python float) of model on n_batches batches
    get_batch(data, batch_size, block_size, generator), in eval mode without gradient; model back in train mode."""
    raise NotImplementedError("estimate_loss_20")


def train_20(model, train_ids, val_ids, settings, generator, block_size=128):
    """Train model in place (contract of Ex B3.20) and return its history: {"train_loss": [...], "lr": [...],
    "val_iter": [...], "val_loss": [...]}."""
    raise NotImplementedError("train_20")

In [ ]:
probes_ok_20 = False
SMALL_20 = {"vocab_size": 88, "block_size": 16, "n_layer": 2, "n_head": 2, "d_model": 16, "dropout": 0.0}
with checked("B3.20"):
    # 1. the two groups of AdamW, parameter by parameter (on the reference model of wb.gpt, so that B3.19 is not needed)
    torch.manual_seed(20)
    small_20 = wbgpt.ReferenceGPT(**SMALL_20)
    opt_20 = configure_optimizer_20(small_20, 1e-3, 0.1)
    if not isinstance(opt_20, torch.optim.AdamW):
        print(f"❌ Ex B3.20 : configure_optimizer_20 doit rendre un torch.optim.AdamW ; reçu {type(opt_20).__name__}.")
        raise StopCheck
    seen_20 = [id(p) for g in opt_20.param_groups for p in g["params"]]
    decay_20 = {id(p): g["weight_decay"] for g in opt_20.param_groups for p in g["params"]}
    wrong_20 = [n for n, p in small_20.named_parameters() if decay_20.get(id(p)) != (0.1 if p.dim() >= 2 else 0.0)]
    groups_ok_20 = sorted(seen_20) == sorted(id(p) for p in small_20.parameters()) and not wrong_20
    verdict("B3.20", groups_ok_20,
            "AdamW : chaque paramètre une fois, weight decay 0,1 sur les matrices et les tables, 0 sur les biais et les layer norms.",
            (f"weight decay mal réglé pour {wrong_20[:4]} : 0,1 sur les paramètres à deux axes ou plus, 0 sur les autres."
             if wrong_20 else "chaque paramètre du modèle doit être dans un groupe, et une seule fois (la tête liée est la "
             "table des tokens)."))
    verdict("B3.20", all(tuple(g["betas"]) == (0.9, 0.99) and g["lr"] == 1e-3 for g in opt_20.param_groups),
            "β = (0,9 ; 0,99) et le learning rate reçu.", "les groupes doivent avoir betas=(0.9, 0.99) et le learning rate reçu.")

    # 2. estimate_loss_20: eval mode and no gradient during the measure, train mode afterwards, the right value
    class Spy_20(nn.Module):
        def __init__(self, inner):
            super().__init__()
            self.inner, self.calls = inner, []

        def forward(self, idx):
            self.calls.append((self.training, torch.is_grad_enabled()))
            return self.inner(idx)

    spy_20 = Spy_20(copy.deepcopy(small_20)).train()
    est_20 = estimate_loss_20(spy_20, DATA.val_ids, 3, 4, 16, torch.Generator().manual_seed(5))
    want_est_20 = wbgpt.estimate_loss(copy.deepcopy(small_20), DATA.val_ids, 3, 4, 16, torch.Generator().manual_seed(5))
    est_ok_20 = type(est_20) is float and math.isclose(est_20, want_est_20, rel_tol=1e-6)
    verdict("B3.20", est_ok_20,
            f"estimate_loss_20 : la moyenne des loss de 3 batches, un float ({fr(est_20, 4)}).",
            f"estimate_loss_20 doit rendre un float Python, la moyenne des loss de 3 batches de get_batch(..., generator) : "
            f"attendu {fr(want_est_20, 4)}, reçu {est_20!r}.")
    mode_ok_20 = len(spy_20.calls) == 3 and all(not t and not g for t, g in spy_20.calls) and spy_20.training
    verdict("B3.20", mode_ok_20,
            "pendant la mesure : mode eval et sans gradient ; le modèle revient en mode train.",
            f"pendant la mesure, attendu 3 passes en mode eval sans gradient, puis le modèle en mode train ; vu "
            f"{[('train' if t else 'eval') + (', gradient' if g else '') for t, g in spy_20.calls]}, mode final "
            f"{'train' if spy_20.training else 'eval'}.")

    # 3. train_20 against the recipe of the contract: 6 iterations of a small model, a clipping that acts at every step
    probe_20 = TRAIN_20 | {"iters": 6, "warmup": 2, "eval_interval": 4, "eval_batches": 2, "batch_size": 4, "clip": 0.05}
    mine_20, ref_20 = copy.deepcopy(small_20), copy.deepcopy(small_20)
    hist_20 = train_20(mine_20, DATA.train_ids, DATA.val_ids, probe_20, torch.Generator().manual_seed(7), block_size=16)
    if not returned("B3.20", "train_20", hist_20):
        raise StopCheck
    want_hist_20 = wbgpt.train(ref_20, DATA.train_ids, DATA.val_ids, probe_20, torch.Generator().manual_seed(7), block_size=16)
    keys_ok_20 = isinstance(hist_20, dict) and all(k in hist_20 for k in want_hist_20)
    hist_ok_20 = keys_ok_20 and len(hist_20["train_loss"]) == len(hist_20["lr"]) == 6 and list(hist_20["val_iter"]) == [0, 4, 6]
    verdict("B3.20", hist_ok_20,
            "historique : 6 loss et 6 learning rates, la validation avant la 1ʳᵉ itération, après la 4ᵉ et après la dernière.",
            "train_20 doit rendre un dict de listes train_loss et lr (une valeur par itération), val_iter et val_loss (avant "
            "la première itération, toutes les eval_interval itérations et après la dernière) ; avec 6 itérations et "
            f"eval_interval = 4, val_iter = [0, 4, 6] ; reçu {hist_20 if not keys_ok_20 else hist_20['val_iter']!r:.200}.")
    if not keys_ok_20:
        raise StopCheck
    lr_ok_20 = len(hist_20["lr"]) == 6 and np.allclose(hist_20["lr"], want_hist_20["lr"], rtol=1e-9, atol=1e-12)
    verdict("B3.20", lr_ok_20,
            "learning rate : montée linéaire de 0, puis cosinus jusqu'à min_lr (ton cosine_lr du ch. 19).",
            f"learning rates attendus {np.round(want_hist_20['lr'], 6).tolist()}, reçus {np.round(hist_20['lr'], 6).tolist()}.")
    same_20 = all(torch.allclose(a, b, rtol=1e-4, atol=1e-7) for a, b in zip(mine_20.parameters(), ref_20.parameters()))
    if same_20:
        print("✅ Ex B3.20 : après 6 itérations, les poids sont ceux de la recette du contrat (AdamW à deux groupes, "
              "learning rate de chaque itération, clipping, un get_batch par itération).")
    else:
        noclip_20 = copy.deepcopy(small_20)
        wbgpt.train(noclip_20, DATA.train_ids, DATA.val_ids, probe_20 | {"clip": 1e9}, torch.Generator().manual_seed(7), block_size=16)
        unclipped_20 = all(torch.allclose(a, b, rtol=1e-4, atol=1e-7) for a, b in zip(mine_20.parameters(), noclip_20.parameters()))
        print("❌ Ex B3.20 : après 6 itérations, tes poids ne sont pas ceux de la recette du contrat : "
              + ("ils sont ceux d'un entraînement sans clipping (la norme des gradients dépasse ici 0,05 à chaque itération)."
                 if unclipped_20 else "vérifie l'ordre de chaque itération (learning rate, get_batch, loss, gradients, "
                 "clipping, pas de l'optimiseur) et que l'évaluation tire ses batches avec un générateur à part."))
    probes_ok_20 = all([groups_ok_20, est_ok_20, mode_ok_20, hist_ok_20, lr_ok_20, same_20])

In [ ]:
model_20 = history_20 = val_char_20 = None
if not (globals().get("probes_ok_20") and globals().get("model_ok_19")):
    print("⏳ Ex B3.20 : l'entraînement attend ton modèle de B3.19 et les contrôles de la cellule précédente.")
else:
    with checked("B3.20"):
        torch.manual_seed(SEED_20)
        trained_20 = MiniGPT_19(**CONFIG_20)
        started_20 = time.perf_counter()
        history_20 = train_20(trained_20, DATA.train_ids, DATA.val_ids, TRAIN_20, torch.Generator().manual_seed(SEED_20))
        seconds_20 = time.perf_counter() - started_20
        if returned("B3.20", "train_20", history_20):
            model_20 = trained_20
            print(f"{TRAIN_20['iters']} itérations en {seconds_20:.0f} s ; dernière loss de validation estimée "
                  f"{fr(history_20['val_loss'][-1], 3)} nats par caractère.")

In [ ]:
if model_20 is None:
    print("⏳ Ex B3.20 : la mesure et la sauvegarde attendent l'entraînement de la cellule précédente.")
else:
    with checked("B3.20"):
        val_char_20 = wbgpt.char_loss(model_20, DATA.val_ids)       # every character of the validation from index 128 on
        path_20 = wbgpt.save_checkpoint(wbgpt.learner_key(FAST_MODE), model_20, history_20, DATA.itos,
                                        extra={"char_loss": val_char_20, "seconds": seconds_20})
        print(f"modèle sauvegardé : {path_20.name}, dans wbgpt.checkpoints_dir()")
        verdict("B3.20", val_char_20 < TARGET_20,
                f"sur toute la validation, {fr(val_char_20, 3)} nats par caractère (perplexité {fr(math.exp(val_char_20), 2)}, "
                f"{fr(val_char_20 / math.log(2), 3)} bits par caractère) : sous {fr(TARGET_20, 2)}.",
                f"sur toute la validation, {fr(val_char_20, 3)} nats par caractère : attendu moins de {fr(TARGET_20, 2)} ; "
                "relance les contrôles de la cellule de vérification, puis ceux de B3.19.")
        rows_20 = [(name, params, budget, loss) for name, params, budget, loss in LSTM_20]
        rows_20.append(("ton mini-GPT (" + ("mode rapide" if FAST_MODE else "mode complet") + ")", n_params(model_20),
                        f"{TRAIN_20['iters']} itérations, {fr(seconds_20 / 60, 1)} min", val_char_20))
        print(f"\n{'modèle':<50} {'paramètres':>10}  {'budget':<38} {'nats/car.':>9} {'perplexité':>10}")
        for name, params, budget, loss in rows_20:
            print(f"{name:<50} {params:>10_}  {budget:<38} {fr(loss, 3):>9} {fr(math.exp(loss), 2):>10}".replace("_", " "))

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une attention à la main, masque causal compris, et dire pourquoi on divise par $\sqrt{d_k}$ ?
2. Sais-tu suivre les formes d'une attention multi-têtes et compter les paramètres d'un GPT ?
3. Sais-tu dire ce que font un encodage sinusoïdal et RoPE, et démontrer la propriété de RoPE ?
4. Sais-tu chiffrer le calcul d'un entraînement et la mémoire d'un KV cache ?

**Pour aller plus loin** : la vidéo « Let's build GPT » d'A. Karpathy et *The Annotated Transformer*, cités dans la fiche. La suite : le bonus B4, les LLM en pratique.

*Les exercices B3.21 à B3.30 (la suite du mini-GPT : courbes, génération et KV cache, cartes d'attention, expériences, ViT, test de causalité et défi) seront ajoutés à la prochaine étape de génération ; relance alors `python tools/start_chapter.py B3` pour obtenir le notebook complet.*